# H3 LocalStudio — Colab SageAttention Linux 修正版 mk22（2026-10-28）
使用本機同一份 Studio：固定 `c77d148a4b1c15c4d859552b56e3949e6f9e3333`、ComfyUI v0.37.0 及本機自訂節點版本。只修改 Colab `/content/LocalStudio`，不會修改 Windows 本機專案。

**新帳號第一次使用：先選 CPU 執行階段，只執行第 0 格**，把所有模型下載到 `MyDrive/H3_Colab/models`（不耗 GPU 時數，需 Drive 約 200 GB 空間），完成後刪除執行階段、改選 GPU，再從第 1 格開始。

先使用擁有 `H3_Colab` 的 Google 帳號登入 Colab，再依序執行各格；不需要修改 ChatGPT 的 Drive 連接，也不需要填 email。需要 **Ampere／Ada／Hopper GPU（例如 A100、L4、H100）與至少 12GB VRAM**；T4 不在此版支援範圍。建議高 RAM 執行階段。安裝 SageAttention 可能需編譯數分鐘。

- 保留文字／首尾圖／Ref2VA、兩種注意力；Ref2VA 仍用 SageAttention。
- mk12：Ref2VA 主模型可在第 1 格用 `REF2VA_QUANT` 切換 INT8 ConvRot（約 19.5 GiB，畫質較接近原始）或 W4A8 mixed（約 11 GiB，省 VRAM）；預設 `auto` 依 VRAM 自動選。
- mk13：新增 **Singularity V2** 主模型切換（文字／首尾圖／參考生影片）與 **VOSR 1× 精修**（畫質提升頁、作品選單，加入生成佇列）。兩者模型若 Drive 缺少，第 5／6 格會自動下載到 Drive 並核對 SHA256（Singularity 約 21.35 GiB、VOSR 約 6.5 GiB）；可在第 1 格 `INSTALL_SINGULARITY`／`INSTALL_VOSR` 關閉。
- mk13：畫質提升方式合併為一個選單（SeedVR2／FlashVSR／VOSR 1×）。
- mk14：各模式預設——文生影片：Singularity V2、8步、sol-attn、480P、約 10 秒；首尾圖生影片：原版 H3、8步、sol-attn、480P、約 10 秒；參考生影片：原版 H3、8步、SageAttention、480P、約 10 秒。聲音皆開啟。
- mk15：三個模式各自記住手動修改的 H3 模型、模式、注意力、解析度、秒數、畫面與聲音；第一次進入用上列預設，切回時恢復該模式上次的值（只在目前網頁有效，重新整理後回到預設）。「再做一次」仍還原作品原設定。
- 視 Drive 中模型提供 Qwen、攝影機視角、影片換角、SeedVR2；修臉節點預設安裝。
- Gemini TTS 可填 API key；Drama3、RTX SDK、Codex／Grok Prompt 優化本版明確停用。
- mk17：模型「用到才複製到 SSD」——ComfyUI 第一次執行到某個 Drive 模型時，先完整複製到 `/content/h3_models` 再從 SSD 載入；之後重新載入都讀 SSD（第 1 格 `LAZY_SSD_CACHE`）。複製進度寫在 `h3_comfy.log`；SSD 不足時改讀 Drive。執行階段刪除後快取清空。
- mk17：參考生影片主模型預設改為 Ref2VA INT8（第 1 格 `REF2VA_QUANT`）。
- mk22：生成佇列的提示框（滑鼠移到執行中的工作上）會即時顯示 ComfyUI 日誌最後 10 行，可看到目前第幾步。Qwen 生圖等成功的圖片（png/jpg/webp）也會自動備份到 Drive；新執行階段 ComfyUI 編號從 00001 重來時，Drive 已有同名舊檔會改名保存（加上時間），不再覆蓋；密碼太短會提示重新輸入，不再中斷。
- mk21：新增第 0 格「CPU 預先下載」，可單獨執行；第 5 格遇到 Drive 缺少的模型一律先下載到 Drive（核對 SHA256、可續傳），不再直接停止。REDQW21 不自動下載。第 0 格在模型齊全時幾秒就略過，之後直接「全部執行」即可。可用第 1 格 `AUTO_DOWNLOAD_MODELS=False` 關閉。
- mk20：換頭／換角的 ComfyUI 節點也接受每步降溫 0 秒（原本伺服器端要求至少 5 秒，Colab 預設 0 會回 400）。
- mk19：換頭／換角（`lib/swap.ts`）與臉部精修（`lib/face-refine.ts`）也改用第 1 格選定的 Ref2VA 主模型，修正 mk18 第 7 格「仍寫死 W4A8 Ref2VA 檔名」的錯誤。
- mk18：選用 Ref2VA INT8 而 Drive 沒有時，第 5 格自動下載到 `models/diffusion_models/`（約 19.5 GiB）並核對 SHA256。
- mk16：開放作品頁（列出本執行階段生成的作品，可用影片操作選單）；刪除與影片延伸仍停用。安全長片轉 480P 保留原檔，避免 Windows 回收桶呼叫。
- 影片在 `/content/h3_outputs` 生成，完成後自動複製影片與 metadata 至 `MyDrive/H3_Colab/outputs`。**圖片、聲音、參考圖不會自動備份。**
- 請先等複製日誌顯示完成，再中斷並刪除執行階段；本機原檔保留至執行階段刪除。

已在 Windows 做靜態與模擬檢查，**尚未在真實 Colab 完成 GPU 生成驗收**；mk13 的 Singularity V2 與 VOSR 僅在 Windows 本機實跑過。Notebook 會實際測試 attention kernel、服務、節點與代理，失敗即停止，不會把 403／404 當成功。

參考：[Colab 使用限制](https://research.google.com/colaboratory/faq.html)、[SageAttention 安裝](https://github.com/thu-ml/SageAttention#installation)。透過網頁生成請使用具有正 CU 餘額的付費執行階段。


In [1]:
# 0. 【CPU 執行階段即可】把所有缺少的模型預先下載到 Google Drive（新帳號第一次使用時跑這格）。
# 只需要這一格：不檢查 GPU、不安裝 Torch/ComfyUI。下載完成後切換成 GPU 執行階段，再從第 1 格開始。
# 已存在的檔案會略過；中斷後重跑會從 .part 續傳；每個檔案都核對 SHA256。
# 模型都齊全時本格只檢查檔案是否存在，幾秒就結束，所以之後按「全部執行」不受影響。
from google.colab import drive
from pathlib import Path
import os, sys, json, shutil, hashlib, subprocess, time
import requests

# ---- 要下載哪些（預設全部） ----
DOWNLOAD_H3_CORE=True        # H3 必要七項（含 Ref2VA INT8 與 W4A8 兩種量化）
DOWNLOAD_SINGULARITY=True    # Singularity V2 主模型＋加速 LoRA（約 21.5 GiB）
DOWNLOAD_QWEN=True           # Qwen Image 2.1 / NoctQ / 攝影機視角 LoRA
DOWNLOAD_SWAP=True           # 換頭／換角 LoRA
DOWNLOAD_SEEDVR2=True        # SeedVR2 畫質提升
DOWNLOAD_EXTRA_LORA=True     # ORB360 等其他 LoRA
DOWNLOAD_FLASHVSR=True       # FlashVSR v1.1
DOWNLOAD_VOSR=True           # VOSR 1× 精修（含 DINOv2 轉檔）
DOWNLOAD_FACE_DETECTOR=True  # 修臉 face_yolov8m.pt
# 不自動下載的模型（REDQW21 在 Civitai 需登入；要用請自行放到 models/diffusion_models/）。
SKIP_MODELS={'REDQW21_V2_int8.safetensors'}

drive.mount('/content/drive')
ROOT=Path('/content')
DRIVE_ROOT=ROOT/'drive/MyDrive/H3_Colab'
MODELS_ROOT=DRIVE_ROOT/'models'
MODELS_ROOT.mkdir(parents=True,exist_ok=True)

CATALOG={'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors': {'folder': 'text_encoders', 'url': 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/e5eb578a89295337b8ff433a035929ce0279e0b6/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'sha256': '35a88d51044231fe332301d7a62aa81e3f2cba62febeb446e2c1e3e0ef76f2c6'}, 'minimax_h3_video_vae_int8_convrot.safetensors': {'folder': 'vae', 'url': 'https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/d8023be02fefbb3633b0cd335c3879f91177299d/minimax_h3_video_vae_int8_convrot.safetensors', 'sha256': '9bb2d96f218c76babd85e0611b85ca8fb330a90546c01a0005e8a58a59593410'}, 'minimax_h3_audio_vae_fp32.safetensors': {'folder': 'vae', 'url': 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/e5eb578a89295337b8ff433a035929ce0279e0b6/vae/minimax_h3_audio_vae_fp32.safetensors', 'sha256': '8e505d95dd1561d47abd43d4238fd40d9bb1ae9e147ed0a4cba778d76ae4db48'}, 'minimax_h3_fl2va_pruned_int8_convrot.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/e5eb578a89295337b8ff433a035929ce0279e0b6/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors', 'sha256': 'e889202c41dafb67b10d67b97f0d8541508036a6090af23425a5c2615d03c47a'}, 'minimax_h3_ref2va_pruned_w4a8_mixed.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/d8023be02fefbb3633b0cd335c3879f91177299d/minimax_h3_ref2va_pruned_w4a8_mixed.safetensors', 'sha256': 'de2c6c29c4ee702b45e48e40daae3834aeee58ab681c732d9152589a87c89910'}, 'minimax_h3_ref2va_pruned_int8_convrot.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors?download=true', 'sha256': '9255f52b6677845ad238f20dfaafa94727053694127ab7f255c048f0f9365779'}, 'minimax_h3_video_vae_fp16.safetensors': {'folder': 'vae', 'url': 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/e5eb578a89295337b8ff433a035929ce0279e0b6/vae/minimax_h3_video_vae_fp16.safetensors', 'sha256': '7c1f131492e7eddacaac9069a61b81bdd39de5cc96561e677c5eab1cdce5e522'}, 'minimax_h3_turbo_v4_step600_ema.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/larryvrh/MiniMax-H3-Turbo-Lora/resolve/43a74557ac3f6539db8e0f2a959d03feb7a81480/minimax_h3_turbo_v4_step600_ema.safetensors', 'sha256': '5f3a626cd72c93a8b9318d6760c510bc5092d2ab13aaba1f932c5bab07a416d3'}, 'qwen_image_2.1_int8_convrot.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/df94239739eef7205973a145ffa6f441f7b64e84/diffusion_models/qwen_image_2.1_int8_convrot.safetensors', 'sha256': 'cb74113cb03faecd79611b01fd7fd642f0aa60d6f0b95086abee214d75eaa57d'}, 'qwen3vl_8b_int8_convrot.safetensors': {'folder': 'text_encoders', 'url': 'https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/df94239739eef7205973a145ffa6f441f7b64e84/text_encoders/qwen3vl_8b_int8_convrot.safetensors', 'sha256': '8bfd0f6e12abf2d2d697ecc888e5e90b0d6741d6708f05799f53afa560452e8f'}, 'qwen_image_2.1_vae_bf16.safetensors': {'folder': 'vae', 'url': 'https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/df94239739eef7205973a145ffa6f441f7b64e84/vae/qwen_image_2.1_vae_bf16.safetensors', 'sha256': 'bb21f7473051e1ac368515dd3f2e15cd44d7a11748ee8823e1ddca3e4876b7c9'}, 'qwen_image_2.1_multiple_angles_v2_step1500.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/akhaliq/Qwen-Image-2.1-Multiple-Angles-LoRA/resolve/0d91c4dfa22b3f5aa89c020eadc30d8f42cb51c0/checkpoints_v2/angles_v2_full_qwen21_multiple_angles_v2_qwen21_multiple_angles_v2_000001500.safetensors', 'sha256': '548c57977c3f1c09c2c9e5f4e83e930cbc6f719c878a576ab8666ce29fef39b6'}, 'minimax_h3_head_swap_v1.0_r32.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/e095edae3f4c69e18b4f27ed9e541078c87a8d98/loras/minimax_h3_head_swap_v1.0_r32.safetensors', 'sha256': 'ea97bd1a8178460661055784d63b4cb560732196fd15d5d7f33bfd214b65ab00'}, 'minimax_h3_character_swap_v1.0_ref2va_r64_gl.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/e095edae3f4c69e18b4f27ed9e541078c87a8d98/loras/minimax_h3_character_swap_v1.0_ref2va_r64_gl.safetensors', 'sha256': '643a3fe63646fee5aebb181a7e1c9f8a1feb5f472709f37e22eeec290cbb75e3'}, 'h3_character_swap_v1.5.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/akatz-ai/MiniMax-H3-Character-Swap-LoRA/resolve/96ea3407b230b6f7df71c58c70e31df96bd7e533/h3_character_swap_v1.5.safetensors', 'sha256': '8f2c3c3eba161df8d0c6099598b4730f641f5ce01e04cd16d2427cff8a35218f'}, 'seedvr2_3b_int8_convrot.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Comfy-Org/SeedVR2/resolve/df48879708206a403d2a61acd55578c2e80fd233/diffusion_models/seedvr2_3b_int8_convrot.safetensors', 'sha256': 'c3dec8bcc5916843a8a858572970597462e1f2dc598d6dfd818f6cd40f53a157'}, 'seedvr2_ema_vae_fp16.safetensors': {'folder': 'vae', 'url': 'https://huggingface.co/Comfy-Org/SeedVR2/resolve/df48879708206a403d2a61acd55578c2e80fd233/vae/seedvr2_ema_vae_fp16.safetensors', 'sha256': '20678548f420d98d26f11442d3528f8b8c94e57ee046ef93dbb7633da8612ca1'}, 'NoctQ_V4_int8_convrot.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Noctaluna/Noct-Q-Uncensored-Qwen-Image-2.1/resolve/main/NoctQ_V4_int8_convrot.safetensors', 'sha256': '4d92d5538253ab36e6a73b056cce5f697950f4303cb9e6686a3198d0ea13f9e8'}, 'REDQW21_V2_int8.safetensors': {'folder': 'diffusion_models', 'url': 'https://civitai.com/api/download/models/3370753?fileId=3258922', 'sha256': '30532b445eabd4b84b108168b6b3fe720535ef400f6bf9d68bd551817d3531a3'}, 'minimax_h3_orb360_step1500.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/MATLOWAI/MiniMax-H3-ORB360-CardSpin/resolve/de8aa4db6f9a0d949855f3dc62948ffc691aa6ce/minimax_h3_orb360_step1500.safetensors', 'sha256': 'e9275b2f456f6bd004c9cb3cf5c636cad87217c5bc96e6a5b5533e917de121da'}}
SINGULARITY_MODEL='Minimax-h3_Singularity_ref2va_v2_pruned_int8.safetensors'
SINGULARITY_LORA='minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors'
SINGULARITY_FILES={
 SINGULARITY_MODEL:{'folder':'diffusion_models','url':'https://huggingface.co/WarmBloodAban/Minimax-h3_Singularity/resolve/1259cf16cd0b655ba691a32fb46a409142762972/Minimax-h3_Singularity_ref2va_v2_pruned_int8.safetensors','sha256':'d7637c2c35cd30facd9e93778c491f6d82f91c4a441aecbdb4c384a62a5f3d9f','gib':19.6},
 SINGULARITY_LORA:{'folder':'loras','url':'https://huggingface.co/lightx2v/Minimax-h3-Turbo/resolve/30f485ed874aa44c8183348ad361c5eabad318da/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors','sha256':'5b9ab5ade15d0775676d01a907268a69a1468dc6033b3b0d3ded5502f3ebb84c','gib':1.9}}
CATALOG.update(SINGULARITY_FILES)

GROUPS={
 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors':'H3_CORE','minimax_h3_video_vae_int8_convrot.safetensors':'H3_CORE',
 'minimax_h3_audio_vae_fp32.safetensors':'H3_CORE','minimax_h3_fl2va_pruned_int8_convrot.safetensors':'H3_CORE',
 'minimax_h3_ref2va_pruned_w4a8_mixed.safetensors':'H3_CORE','minimax_h3_ref2va_pruned_int8_convrot.safetensors':'H3_CORE',
 'minimax_h3_video_vae_fp16.safetensors':'H3_CORE','minimax_h3_turbo_v4_step600_ema.safetensors':'H3_CORE',
 'qwen_image_2.1_int8_convrot.safetensors':'QWEN','qwen3vl_8b_int8_convrot.safetensors':'QWEN','qwen_image_2.1_vae_bf16.safetensors':'QWEN',
 'qwen_image_2.1_multiple_angles_v2_step1500.safetensors':'QWEN','NoctQ_V4_int8_convrot.safetensors':'QWEN','REDQW21_V2_int8.safetensors':'QWEN',
 'minimax_h3_head_swap_v1.0_r32.safetensors':'SWAP','minimax_h3_character_swap_v1.0_ref2va_r64_gl.safetensors':'SWAP','h3_character_swap_v1.5.safetensors':'SWAP',
 'seedvr2_3b_int8_convrot.safetensors':'SEEDVR2','seedvr2_ema_vae_fp16.safetensors':'SEEDVR2',
 'minimax_h3_orb360_step1500.safetensors':'EXTRA_LORA',
 SINGULARITY_MODEL:'SINGULARITY',SINGULARITY_LORA:'SINGULARITY'}

def download_file(url,target,sha256=None,attempts=4):
    """Stream into <target>.part while hashing; resume with HTTP Range after a drop; rename only when complete and verified."""
    target=Path(target); target.parent.mkdir(parents=True,exist_ok=True)
    part=target.with_name(target.name+'.part')
    for attempt in range(1,attempts+1):
        try:
            h=hashlib.sha256(); done=0
            if part.exists():
                with part.open('rb') as f:
                    for chunk in iter(lambda:f.read(16*1024*1024),b''): h.update(chunk); done+=len(chunk)
            headers={'Range':f'bytes={done}-'} if done else {}
            with requests.get(url,headers=headers,stream=True,timeout=(30,300),allow_redirects=True) as r:
                if done and r.status_code==200:
                    h=hashlib.sha256(); done=0; part.unlink()  # server ignored Range: start over
                if not (r.status_code in (200,206) or (done and r.status_code==416)):
                    raise OSError(f'HTTP {r.status_code}')
                if r.status_code!=416:  # 416: .part already holds the whole file
                    total=done+int(r.headers.get('content-length',0) or 0); mark=done
                    with part.open('ab') as f:
                        for chunk in r.iter_content(16*1024*1024):
                            f.write(chunk); h.update(chunk); done+=len(chunk)
                            if done-mark>=1024**3:
                                mark=done; print(f'  {target.name}: {done/1024**3:.1f}/{total/1024**3:.1f} GiB',flush=True)
            digest=h.hexdigest()
            if sha256 and digest.lower()!=sha256.lower():
                part.unlink(missing_ok=True)
                raise ValueError(f'SHA256 不符，已刪除下載檔：{target.name}')
            os.replace(part,target)
            print('下載完成:',target,flush=True)
            return digest
        except ValueError:
            raise
        except Exception as exc:
            print(f'  下載中斷（第 {attempt}/{attempts} 次），稍後續傳：',type(exc).__name__,str(exc)[:200],flush=True)
            if attempt==attempts: raise
            time.sleep(10*attempt)

def remote_size(url):
    """Size from HEAD (follows redirects); 0 if the server does not say."""
    try:
        r=requests.head(url,allow_redirects=True,timeout=60)
        size=int(r.headers.get('content-length',0) or 0)
        if not size and r.history: size=int(r.history[0].headers.get('x-linked-size',0) or 0)
        return size
    except requests.RequestException:
        return 0

def make_local_room(size):
    """Drive for desktop caches writes on the VM disk before upload. When the local disk runs low,
    flush (blocks until everything is uploaded) and remount so the cache is freed."""
    if shutil.disk_usage('/').free < size+10*1024**3:
        print('本機暫存空間不足，等待 Drive 上傳完成後再繼續…',flush=True)
        drive.flush_and_unmount(); drive.mount('/content/drive')

# ---- 列出要下載的檔案 ----
plan=[]
for name,item in CATALOG.items():
    if not globals()['DOWNLOAD_'+GROUPS.get(name,'EXTRA_LORA')]: continue
    target=MODELS_ROOT/item['folder']/name
    if target.is_file(): continue
    if name in SKIP_MODELS: continue
    plan.append((name,item,target))
sizes={}
if plan:
    print(f'需要下載 {len(plan)} 個檔案，正在查詢大小…',flush=True)
    sizes={name:remote_size(item['url']) for name,item,_ in plan}
    total=sum(sizes.values())/1024**3
    try: drive_free=shutil.disk_usage(DRIVE_ROOT).free/1024**3
    except OSError: drive_free=float('inf')
    for name,_,_ in plan: print(f'  {sizes[name]/1024**3:6.2f} GiB  {name}')
    print(f'合計約 {total:.1f} GiB（另有 FlashVSR／VOSR／修臉偵測器約 12 GiB）；Drive 剩餘約 {drive_free:.1f} GiB',flush=True)
    assert drive_free>total+2, 'Drive 空間不足，請清出空間或把上方 DOWNLOAD_* 關掉一些再重跑'
downloaded=bool(plan)

failed=[]
for i,(name,item,target) in enumerate(plan,1):
    print(f'[{i}/{len(plan)}] 下載 {name}',flush=True)
    try:
        make_local_room(sizes[name])
        download_file(item['url'],target,item.get('sha256'))
    except Exception as exc:
        failed.append(name); print('  ✗ 失敗：',type(exc).__name__,str(exc)[:200],flush=True)

if DOWNLOAD_FACE_DETECTOR:
    detector=MODELS_ROOT/'ultralytics/bbox/face_yolov8m.pt'
    if not detector.is_file():
        downloaded=True
        try: download_file('https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt',detector,'717923c19b3f4bbf5250b728f1fa6b2cb72a33aed1d236ea9caf0e21ad943e5f')
        except Exception as exc: failed.append('face_yolov8m.pt'); print('  ✗',exc)

if DOWNLOAD_FLASHVSR:
    need=['LQ_proj_in.ckpt','TCDecoder.ckpt','diffusion_pytorch_model_streaming_dmd.safetensors','Wan2.1_VAE.pth']
    drive_fv=MODELS_ROOT/'FlashVSR-v1.1'
    if not all((drive_fv/n).is_file() for n in need):
        print('下載 FlashVSR v1.1…',flush=True); downloaded=True
        try:
            make_local_room(8*1024**3)
            subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub'],check=True)
            from huggingface_hub import snapshot_download
            snapshot_download(repo_id='JunhaoZhuang/FlashVSR-v1.1',local_dir=str(drive_fv),allow_patterns=need)
            assert all((drive_fv/n).is_file() for n in need)
        except Exception as exc: failed.append('FlashVSR'); print('  ✗',type(exc).__name__,str(exc)[:200])

if DOWNLOAD_VOSR:
    hf='https://huggingface.co/CSWRY/VOSR/resolve/f24b3061b7f350b81e1907bdcfc27f71fb4ff3f3/'
    bundle=MODELS_ROOT/'vosr2/VOSR2'
    vosr_files=[('VOSR2/args.json','args.json',None),
                ('VOSR2/checkpoints/ema_model.safetensors','checkpoints/ema_model.safetensors','bdcaa81e4c675b6074de643e27daafe73eb8125d4d0264d1bf30a004e9644b71'),
                ('Qwen-Image-vae-2d/config.json','Qwen-Image-vae-2d/config.json',None),
                ('Qwen-Image-vae-2d/diffusion_pytorch_model.safetensors','Qwen-Image-vae-2d/diffusion_pytorch_model.safetensors','1dd2c67f3f29734d48709513ed181d4f351966d76ba95f77a5cff8e1daf547fa')]
    try:
        for remote,rel,sha in vosr_files:
            if not (bundle/rel).is_file():
                print('下載 VOSR:',rel,flush=True); make_local_room(6*1024**3); downloaded=True
                download_file(hf+remote,bundle/rel,sha)
        dino=bundle/'dinov2_vitl14.safetensors'
        if not dino.is_file():
            downloaded=True
            # Same conversion as cell 6, using the Colab kernel's CPU torch.
            pth=ROOT/'h3_vosr_tmp/dinov2_vitl14_pretrain.pth'
            download_file(hf+'torch_cache/checkpoints/dinov2_vitl14_pretrain.pth',pth,'d5383ea8f4877b2472eb973e0fd72d557c7da5d3611bd527ceeb1d7162cbf428')
            local=pth.with_name('dinov2_vitl14.safetensors')
            subprocess.run([sys.executable,'-c','import sys,torch\nfrom safetensors.torch import save_file\nstate=torch.load(sys.argv[1],map_location="cpu",weights_only=True,mmap=True)\nsave_file({k:v.contiguous() for k,v in state.items()},sys.argv[2])',str(pth),str(local)],check=True)
            part=dino.with_name(dino.name+'.part'); shutil.copyfile(local,part); os.replace(part,dino)
            shutil.rmtree(pth.parent)
            print('DINOv2 已轉換並存入 Drive',flush=True)
    except Exception as exc: failed.append('VOSR'); print('  ✗',type(exc).__name__,str(exc)[:200])

if not downloaded:
    print('✅ 模型都已在 Drive，略過下載。')
else:
    print('等待所有檔案上傳到 Google Drive（大檔可能要幾分鐘）…',flush=True)
    drive.flush_and_unmount(); drive.mount('/content/drive')
if not downloaded: pass
elif failed:
    print('⚠ 以下項目失敗，可直接重跑本格續傳：',', '.join(failed))
else:
    print('✅ 全部下載完成並已上傳 Drive。若目前是 CPU 執行階段：請「中斷連線並刪除執行階段」，改選 GPU（L4／A100）後再全部執行。')


Mounted at /content/drive
✅ 模型都已在 Drive，略過下載。


In [2]:
# 1. 先確認 Colab 右上角登入的是擁有 H3_Colab 的帳號，再掛載 Drive、設定。缺少的模型第 5 格會自動下載到 Drive（建議先用第 0 格在 CPU 執行階段下載好）。
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, json, getpass

drive.mount('/content/drive')
ROOT=Path('/content')
DRIVE_ROOT=ROOT/'drive/MyDrive/H3_Colab'
MODELS_ROOT=DRIVE_ROOT/'models'
DRIVE_OUTPUTS_ROOT=DRIVE_ROOT/'outputs'
OUTPUTS_ROOT=ROOT/'h3_outputs'
REPO=ROOT/'LocalStudio'; COMFY=ROOT/'ComfyUI'; WEB=REPO/'h3-local-studio'
VENV=ROOT/'h3_venv'; PYTHON=VENV/'bin/python'
# False (default): skip copying many GB on every Colab launch. Set True for optional SSD caching.
COPY_MODELS_TO_SSD=False
# mk17 (default): copy a Drive model to the local SSD the first time a generation loads it, then load from SSD.
# Only used models are copied; if the SSD is short on space that model is read from Drive. Ignored when COPY_MODELS_TO_SSD=True.
LAZY_SSD_CACHE=True
# True performs complete SHA256 reads; large Drive files can take considerable time.
VERIFY_MODEL_SHA256=False
# mk21：True（預設）時第 5 格會把 Drive 缺少的所有模型自動下載到 Drive。
AUTO_DOWNLOAD_MODELS=True
INSTALL_FACE_REFINE=True
# Ref2VA（參考圖生影片）主模型量化：兩個檔都可放在 MyDrive/H3_Colab/models/diffusion_models/。
#   'auto'：VRAM ≥ 40GB（A100 40/80G、H100）用 int8，其餘（L4 24G）用 w4a8；偏好的檔不在 Drive 時改用另一個並提示。
#   'int8'：minimax_h3_ref2va_pruned_int8_convrot（約 19.5 GiB，畫質較好、吃 VRAM；Drive 沒有時第 5 格自動下載）
#   'w4a8'：minimax_h3_ref2va_pruned_w4a8_mixed（約 11 GiB，L4 較不易 offload）
# 改這個值之後要重跑第 5、7 格（第 7 格會重新 build），快速恢復格不會套用新設定。
REF2VA_QUANT='int8'
assert REF2VA_QUANT in ('auto','int8','w4a8'), "REF2VA_QUANT 只能是 'auto'、'int8'、'w4a8'"
# Optional faster 畫質提升 engine. Model: MyDrive/H3_Colab/models/FlashVSR-v1.1 (auto-downloaded there once if missing).
INSTALL_FLASHVSR=True
# mk13：Singularity V2 主模型＋專用加速 LoRA（約 21.35 GiB）。Drive 缺檔時自動下載到 MyDrive/H3_Colab/models/ 並核對 SHA256。
INSTALL_SINGULARITY=True
# mk13：VOSR 1× 精修（約 6.5 GiB）。Drive 缺檔時自動下載到 MyDrive/H3_Colab/models/vosr2/VOSR2/ 並核對 SHA256。
INSTALL_VOSR=True
BASIC_USER='h3'
while True:
    BASIC_PASSWORD=getpass.getpass('設定 Studio 密碼（至少 8 字元，不存進 Drive；登入帳號是 h3）：')
    if len(BASIC_PASSWORD)>=8: break
    print(f'⚠ 密碼只有 {len(BASIC_PASSWORD)} 個字元，至少要 8 個，請重新輸入。',flush=True)
GEMINI_KEY=getpass.getpass('Gemini TTS API key（選填，直接 Enter 略過）：').strip()
for folder in (MODELS_ROOT,DRIVE_OUTPUTS_ROOT,OUTPUTS_ROOT): folder.mkdir(parents=True,exist_ok=True)
print('模型根目錄:',MODELS_ROOT)
print('生成位置:',OUTPUTS_ROOT,'→ 完成後複製到',DRIVE_OUTPUTS_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
設定 Studio 密碼（至少 8 字元，不存進 Drive；登入帳號是 h3）：··········
Gemini TTS API key（選填，直接 Enter 略過）：··········
模型根目錄: /content/drive/MyDrive/H3_Colab/models
生成位置: /content/h3_outputs → 完成後複製到 /content/drive/MyDrive/H3_Colab/outputs


In [3]:
# 2. 系統工具與獨立 Python 環境，不改動 Colab kernel 的 Torch。
import shutil, signal, time, urllib.request, hashlib

def run(args,**kwargs):
    subprocess.run([str(x) for x in args],check=True,**kwargs)

def process_token(pid):
    try:
        fields=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
        return None if fields[0]=='Z' else fields[19]
    except (FileNotFoundError,ProcessLookupError):
        return None

def stop_managed(name):
    record=ROOT/(name+'.pid.json')
    if not record.exists(): return
    saved=json.loads(record.read_text()); pid=saved['pid']
    if process_token(pid)==saved['token']:
        os.killpg(pid,signal.SIGTERM)
        for _ in range(50):
            if process_token(pid)!=saved['token']: break
            time.sleep(0.1)
        if process_token(pid)==saved['token']: os.killpg(pid,signal.SIGKILL)
    record.unlink(missing_ok=True)

# Only stop processes previously launched and identified by this notebook.
for name in ('h3_cloudflared','h3_drive_copy','h3_studio','h3_comfy'): stop_managed(name)
run(['apt-get','update','-qq'])
run(['apt-get','install','-y','-qq','ffmpeg','git','curl','nginx','apache2-utils','ca-certificates','build-essential','ninja-build','python3-venv','libgl1','libglib2.0-0'])
major=int(subprocess.check_output(['node','--version'],text=True).strip().lstrip('v').split('.')[0]) if shutil.which('node') else 0
if major!=24:
    run(['bash','-o','pipefail','-c','curl -fsSL https://deb.nodesource.com/setup_24.x | bash -'])
    run(['apt-get','install','-y','-qq','nodejs'])
# Colab system Python can be 3.13 and python3-venv/ensurepip may not match it.
# Install isolated Python 3.11 via uv, then seed pip in the venv.
# This avoids modifying Colab kernel Torch and works across Python 3.12/3.13 images.
run([sys.executable,'-m','pip','install','-q','uv'])
UV=[sys.executable,'-m','uv']
run([*UV,'python','install','3.11'])
if PYTHON.exists():
    valid=subprocess.run([str(PYTHON),'-m','pip','--version'],capture_output=True,text=True)
    version=subprocess.run([str(PYTHON),'-c','import sys; print(sys.version_info[:2])'],capture_output=True,text=True)
    if valid.returncode!=0 or version.returncode!=0 or '(3, 11)' not in version.stdout:
        print('Removing incomplete or incompatible venv:',VENV,flush=True)
        assert VENV.resolve().is_relative_to(ROOT)
        shutil.rmtree(VENV)
if not PYTHON.exists():
    run([*UV,'venv','--python','3.11','--seed',str(VENV)])
run([PYTHON,'-m','pip','install','--upgrade','pip','setuptools','wheel','packaging','ninja'])
print('venv Python:',subprocess.check_output([str(PYTHON),'--version'],text=True).strip(),flush=True)
cloud_url='https://github.com/cloudflare/cloudflared/releases/download/2026.10.3/cloudflared-linux-amd64'
cloud=Path('/usr/local/bin/cloudflared')
expected='053b80b0a23031a038fdbf3c073f59f82b8f1f152aa67971d7383ffafc0dbab0'
if not cloud.exists() or hashlib.sha256(cloud.read_bytes()).hexdigest()!=expected:
    run(['curl','--fail','--location','--retry','3',cloud_url,'-o',cloud])
assert hashlib.sha256(cloud.read_bytes()).hexdigest()==expected, 'cloudflared SHA256 mismatch'
cloud.chmod(0o755)
print('獨立 Python:',PYTHON)


venv Python: Python 3.11.18
獨立 Python: /content/h3_venv/bin/python


In [4]:
# 3. 固定原始碼版本；重新執行會回復 Colab checkout 後再套用下一格的補丁。
# 僅重置這份 notebook 所管理的 /content 目錄。
STUDIO_COMMIT='c77d148a4b1c15c4d859552b56e3949e6f9e3333'
COMFY_COMMIT='73c9bad4d21e7addbe1d13bc92eee0f1431b017d'
def checkout(url,path,commit,patch_files=()):
    path=Path(path)
    assert path.resolve().is_relative_to(ROOT) and not path.resolve().is_relative_to(DRIVE_ROOT)
    if not (path/'.git').exists(): run(['git','clone',url,path])
    actual=subprocess.check_output(['git','-C',path,'remote','get-url','origin'],text=True).strip()
    assert actual.rstrip('/').removesuffix('.git')==url.rstrip('/').removesuffix('.git'), f'Unexpected repo at {path}'
    run(['git','-C',path,'fetch','origin',commit])
    present=[f for f in patch_files if (path/f).exists()]
    if present: run(['git','-C',path,'restore','--',*present])
    run(['git','-C',path,'checkout','--detach',commit])
    assert subprocess.check_output(['git','-C',path,'rev-parse','HEAD'],text=True).strip()==commit

checkout('https://github.com/shellpig/LocalStudio.git',REPO,STUDIO_COMMIT,
         ['h3-local-studio/app/page.tsx','h3-local-studio/lib/comfy.ts','h3-local-studio/lib/tts.ts','h3-local-studio/lib/drama3.ts','h3-local-studio/app/speech-studio.tsx','h3-local-studio/app/swap-studio.tsx','integrations/ComfyUI-H3-Studio/h3_studio_api.py','integrations/ComfyUI-H3-Studio/h3_studio_enhance.py','h3-local-studio/lib/enhance.ts','h3-local-studio/app/enhance-studio.tsx','h3-local-studio/lib/face-refine.ts','h3-local-studio/app/generation-queue-panel.tsx','h3-local-studio/lib/vosr.ts','integrations/ComfyUI-H3-Studio/h3_studio_vosr.py','integrations/ComfyUI-H3-Studio/h3_studio_swap.py','h3-local-studio/lib/swap.ts'])
checkout('https://github.com/comfyanonymous/ComfyUI.git',COMFY,COMFY_COMMIT)
for url,name,commit in [
 ('Larryvrh/ComfyUI-MiniMax-H3-Turbo','ComfyUI-MiniMax-H3-Turbo','55fee864dd7b2976b1c4ce3c3d5f7968f181409f'),
 ('NikoDemon80/ComfyUI-H3-Motion-Context','ComfyUI-H3-Motion-Context','c140ae99b8c38f782ebd8564c267b42aacade6a4'),
 ('vizart-vj/ComfyUI-MiniMax-H3-LongMedia','ComfyUI-MiniMax-H3-LongMedia','96c50f7febc8c2e80c2a50d030d6cb3ff3de7ee3'),
 ('kijai/ComfyUI-KJNodes','ComfyUI-KJNodes','073efb07419f56cc714e099a82e49fbc23ad9263')]:
    checkout('https://github.com/'+url+'.git',COMFY/'custom_nodes'/name,commit)
link=COMFY/'custom_nodes/ComfyUI-H3-Studio'
if link.is_symlink():
    assert link.resolve()==(REPO/'integrations/ComfyUI-H3-Studio').resolve()
elif link.exists():
    raise RuntimeError(f'{link} already exists and is not the expected symlink')
else:
    link.symlink_to(REPO/'integrations/ComfyUI-H3-Studio',target_is_directory=True)


In [5]:
# 4. Linux Torch + SageAttention。編譯失敗會停止，不會假裝已啟用加速。
import re
assert shutil.which('nvidia-smi'), '請先選擇 GPU 執行階段'
run(['nvidia-smi'])
assert shutil.which('nvcc'), '此執行階段缺少 CUDA 編譯工具，無法安裝 SageAttention'
nvcc=subprocess.check_output(['nvcc','--version'],text=True)
m=re.search(r'release (\d+)\.(\d+)',nvcc)
assert m, nvcc
cuda=tuple(map(int,m.groups()))
assert cuda>=(12,8), '需要 CUDA toolkit 12.8 或更新的 Colab 執行階段'
index='cu130' if cuda[0]>=13 else 'cu128'
run([PYTHON,'-m','pip','install','torch==2.10.0','torchvision==0.25.0','torchaudio==2.10.0','--index-url','https://download.pytorch.org/whl/'+index])
run([PYTHON,'-m','pip','install','-r',COMFY/'requirements.txt'])
run([PYTHON,'-m','pip','install','-r',COMFY/'custom_nodes/ComfyUI-KJNodes/requirements.txt'])
probe="""
import torch,psutil
assert torch.cuda.is_available(), 'Torch CUDA 未就緒；檢查 Colab 驅動與 wheel 是否相容'
p=torch.cuda.get_device_properties(0)
assert torch.cuda.get_device_capability(0) in [(8,0),(8,6),(8,9),(9,0)], '此 notebook 支援 A100/Ampere、L4/Ada、H100/Hopper，請更換 GPU'
assert p.total_memory>=11.5*1024**3, '需要至少 12GB VRAM'
print(p.name, round(p.total_memory/1024**3,1),'GiB VRAM; RAM',round(psutil.virtual_memory().total/1024**3,1),'GiB')
if psutil.virtual_memory().total<28*1024**3: print('注意：低於 28GiB RAM，請優先改用高 RAM 執行階段以減少換頁')
"""
run([PYTHON,'-c',probe])
# SageAttention v2.2.0 is NOT reliably available as a PyPI sdist.
# Prefer a version-matched Linux wheel for the supported Python 3.11 + Torch 2.10 + CUDA 13 stack.
# Never silently disable SageAttention: the original Studio defaults to it.
import sys, subprocess, os, time
sage_log=ROOT/'h3_sage_install.log'
env=os.environ.copy(); env.update(MAX_JOBS='2',EXT_PARALLEL='1',NVCC_APPEND_FLAGS='--threads 2')

def checked_install(cmd, label):
    print('▶',label,flush=True)
    with sage_log.open('a',encoding='utf-8') as f:
        f.write('\n=== '+label+' ===\n')
        f.flush()
        p=subprocess.Popen([str(x) for x in cmd],env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        tail=[]
        for line in p.stdout:
            f.write(line); f.flush()
            tail.append(line.rstrip())
            if len(tail)>70: tail.pop(0)
            if any(k in line.lower() for k in ('error:', 'failed', 'building wheel', 'downloading', 'successfully installed')):
                print(line.rstrip(),flush=True)
        code=p.wait()
    if code:
        print('\nSageAttention 安裝失敗，最後 70 行：\n'+'\n'.join(tail),flush=True)
        raise RuntimeError(f'{label} 失敗（exit {code}），完整日誌：{sage_log}。不要重新安裝整份 Notebook。')

if index == 'cu130':
    sage_wheel=('https://github.com/deepbeepmeep/SageAttention/releases/download/'
                'v2.2.0-linux.cg1/sageattention-2.2.0+cu130torch2.10.0.cg1-cp311-cp311-linux_x86_64.whl')
    checked_install([PYTHON,'-m','pip','install','--no-deps',sage_wheel],
                    'SageAttention 2.2.0 預編譯 Linux wheel (Py3.11/Torch2.10/CUDA13)')
else:
    # CUDA 12.8: build from the upstream GitHub source, not `pip install sageattention==2.2.0`.
    sage_src=ROOT/'SageAttention_build'
    if not (sage_src/'.git').exists():
        run(['git','clone','--depth','1','--branch','v2.2.0','https://github.com/thu-ml/SageAttention.git',sage_src])
    checked_install([PYTHON,'-m','pip','install','--no-build-isolation','--no-deps',str(sage_src)],
                    'SageAttention 2.2.0 CUDA12.8 原始碼編譯')

# Verify the exact core symbols KJNodes imports, and execute real GPU kernels.
probe="""
import torch,comfy_kitchen as ck
from sageattention import sageattn
from sageattention.core import per_thread_int8_triton,per_warp_int8_cuda,per_block_int8_triton,per_channel_fp8,get_cuda_arch_versions,attn_false
q=torch.randn(1,2,512,128,device='cuda',dtype=torch.bfloat16)
y=sageattn(q,q.clone(),q.clone(),tensor_layout='HND',is_causal=False)
torch.cuda.synchronize(); assert y.shape==q.shape and torch.isfinite(y).all()
assert ck.sol_attn_is_available(q.device), 'Comfy Kitchen sol-attn kernel unavailable'
assert ck.int8_attention_is_available(q.device), 'Comfy Kitchen INT8 attention unavailable'
z=ck.sol_attn(q.transpose(1,2).contiguous(),q.transpose(1,2).contiguous(),q.transpose(1,2).contiguous(),tau=1.3)
torch.cuda.synchronize(); assert torch.isfinite(z).all()
print('SageAttention and Kitchen sol-attn GPU checks passed:',get_cuda_arch_versions())
"""
run([PYTHON,'-c',probe])
run(['npm','ci'],cwd=WEB)


▶ SageAttention 2.2.0 預編譯 Linux wheel (Py3.11/Torch2.10/CUDA13)


## 模型位置與缺檔檢查
你截圖中的四個資料夾正確。下格會列出每個檔名應放在 `MyDrive/H3_Colab/models/<分類>/` 的哪裡。
H3 主要七項需齊全（其中 Ref2VA 主模型依 `REF2VA_QUANT` 只需 INT8 或 W4A8 其中一個）；其他模型有放才開放對應功能。修臉偵測器另放 `models/ultralytics/bbox/face_yolov8m.pt`，安裝格會下載並核對 SHA256，不必先手動上傳。

**mk21：缺檔一律自動下載到 Drive**，下載時同步計算 SHA256，中斷後重跑該格會續傳。連結固定採本機 README 來源；REDQW21 不自動下載；要用請自行從 Civitai 下載並改名成 `REDQW21_V2_int8.safetensors` 放到 `diffusion_models/`（不是 LoRA）。

**快速檢查：**預設僅讀取檔名、檔案大小與 safetensors 標頭；完整 SHA256 由第 1 格 `VERIFY_MODEL_SHA256=True` 才啟用。驗證資訊存於 `MyDrive/H3_Colab/configs/model_validation_cache_v1.json`，大小或修改時間改變時不沿用舊 SHA256。預設不把所有模型複製到 SSD。


In [6]:
# 5. 缺少的模型先自動下載到 Drive（mk21），再快速檢查：檔名／大小／safetensors 標頭；完整 SHA256 為選用。
import struct, json, os, hashlib, shutil, time
from pathlib import Path
CATALOG={'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors': {'folder': 'text_encoders', 'url': 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/e5eb578a89295337b8ff433a035929ce0279e0b6/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'sha256': '35a88d51044231fe332301d7a62aa81e3f2cba62febeb446e2c1e3e0ef76f2c6'}, 'minimax_h3_video_vae_int8_convrot.safetensors': {'folder': 'vae', 'url': 'https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/d8023be02fefbb3633b0cd335c3879f91177299d/minimax_h3_video_vae_int8_convrot.safetensors', 'sha256': '9bb2d96f218c76babd85e0611b85ca8fb330a90546c01a0005e8a58a59593410'}, 'minimax_h3_audio_vae_fp32.safetensors': {'folder': 'vae', 'url': 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/e5eb578a89295337b8ff433a035929ce0279e0b6/vae/minimax_h3_audio_vae_fp32.safetensors', 'sha256': '8e505d95dd1561d47abd43d4238fd40d9bb1ae9e147ed0a4cba778d76ae4db48'}, 'minimax_h3_fl2va_pruned_int8_convrot.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/e5eb578a89295337b8ff433a035929ce0279e0b6/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors', 'sha256': 'e889202c41dafb67b10d67b97f0d8541508036a6090af23425a5c2615d03c47a'}, 'minimax_h3_ref2va_pruned_w4a8_mixed.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/d8023be02fefbb3633b0cd335c3879f91177299d/minimax_h3_ref2va_pruned_w4a8_mixed.safetensors', 'sha256': 'de2c6c29c4ee702b45e48e40daae3834aeee58ab681c732d9152589a87c89910'}, 'minimax_h3_ref2va_pruned_int8_convrot.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors?download=true', 'sha256': '9255f52b6677845ad238f20dfaafa94727053694127ab7f255c048f0f9365779'}, 'minimax_h3_video_vae_fp16.safetensors': {'folder': 'vae', 'url': 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/e5eb578a89295337b8ff433a035929ce0279e0b6/vae/minimax_h3_video_vae_fp16.safetensors', 'sha256': '7c1f131492e7eddacaac9069a61b81bdd39de5cc96561e677c5eab1cdce5e522'}, 'minimax_h3_turbo_v4_step600_ema.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/larryvrh/MiniMax-H3-Turbo-Lora/resolve/43a74557ac3f6539db8e0f2a959d03feb7a81480/minimax_h3_turbo_v4_step600_ema.safetensors', 'sha256': '5f3a626cd72c93a8b9318d6760c510bc5092d2ab13aaba1f932c5bab07a416d3'}, 'qwen_image_2.1_int8_convrot.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/df94239739eef7205973a145ffa6f441f7b64e84/diffusion_models/qwen_image_2.1_int8_convrot.safetensors', 'sha256': 'cb74113cb03faecd79611b01fd7fd642f0aa60d6f0b95086abee214d75eaa57d'}, 'qwen3vl_8b_int8_convrot.safetensors': {'folder': 'text_encoders', 'url': 'https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/df94239739eef7205973a145ffa6f441f7b64e84/text_encoders/qwen3vl_8b_int8_convrot.safetensors', 'sha256': '8bfd0f6e12abf2d2d697ecc888e5e90b0d6741d6708f05799f53afa560452e8f'}, 'qwen_image_2.1_vae_bf16.safetensors': {'folder': 'vae', 'url': 'https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/df94239739eef7205973a145ffa6f441f7b64e84/vae/qwen_image_2.1_vae_bf16.safetensors', 'sha256': 'bb21f7473051e1ac368515dd3f2e15cd44d7a11748ee8823e1ddca3e4876b7c9'}, 'qwen_image_2.1_multiple_angles_v2_step1500.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/akhaliq/Qwen-Image-2.1-Multiple-Angles-LoRA/resolve/0d91c4dfa22b3f5aa89c020eadc30d8f42cb51c0/checkpoints_v2/angles_v2_full_qwen21_multiple_angles_v2_qwen21_multiple_angles_v2_000001500.safetensors', 'sha256': '548c57977c3f1c09c2c9e5f4e83e930cbc6f719c878a576ab8666ce29fef39b6'}, 'minimax_h3_head_swap_v1.0_r32.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/e095edae3f4c69e18b4f27ed9e541078c87a8d98/loras/minimax_h3_head_swap_v1.0_r32.safetensors', 'sha256': 'ea97bd1a8178460661055784d63b4cb560732196fd15d5d7f33bfd214b65ab00'}, 'minimax_h3_character_swap_v1.0_ref2va_r64_gl.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/e095edae3f4c69e18b4f27ed9e541078c87a8d98/loras/minimax_h3_character_swap_v1.0_ref2va_r64_gl.safetensors', 'sha256': '643a3fe63646fee5aebb181a7e1c9f8a1feb5f472709f37e22eeec290cbb75e3'}, 'h3_character_swap_v1.5.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/akatz-ai/MiniMax-H3-Character-Swap-LoRA/resolve/96ea3407b230b6f7df71c58c70e31df96bd7e533/h3_character_swap_v1.5.safetensors', 'sha256': '8f2c3c3eba161df8d0c6099598b4730f641f5ce01e04cd16d2427cff8a35218f'}, 'seedvr2_3b_int8_convrot.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Comfy-Org/SeedVR2/resolve/df48879708206a403d2a61acd55578c2e80fd233/diffusion_models/seedvr2_3b_int8_convrot.safetensors', 'sha256': 'c3dec8bcc5916843a8a858572970597462e1f2dc598d6dfd818f6cd40f53a157'}, 'seedvr2_ema_vae_fp16.safetensors': {'folder': 'vae', 'url': 'https://huggingface.co/Comfy-Org/SeedVR2/resolve/df48879708206a403d2a61acd55578c2e80fd233/vae/seedvr2_ema_vae_fp16.safetensors', 'sha256': '20678548f420d98d26f11442d3528f8b8c94e57ee046ef93dbb7633da8612ca1'}, 'NoctQ_V4_int8_convrot.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Noctaluna/Noct-Q-Uncensored-Qwen-Image-2.1/resolve/main/NoctQ_V4_int8_convrot.safetensors', 'sha256': '4d92d5538253ab36e6a73b056cce5f697950f4303cb9e6686a3198d0ea13f9e8'}, 'REDQW21_V2_int8.safetensors': {'folder': 'diffusion_models', 'url': 'https://civitai.com/api/download/models/3370753?fileId=3258922', 'sha256': '30532b445eabd4b84b108168b6b3fe720535ef400f6bf9d68bd551817d3531a3'}, 'minimax_h3_orb360_step1500.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/MATLOWAI/MiniMax-H3-ORB360-CardSpin/resolve/de8aa4db6f9a0d949855f3dc62948ffc691aa6ce/minimax_h3_orb360_step1500.safetensors', 'sha256': 'e9275b2f456f6bd004c9cb3cf5c636cad87217c5bc96e6a5b5533e917de121da'}}
# Ref2VA main model: chosen by REF2VA_QUANT (cell 1). Only the chosen one is required.
REF2VA_FILES={'int8':'minimax_h3_ref2va_pruned_int8_convrot.safetensors','w4a8':'minimax_h3_ref2va_pruned_w4a8_mixed.safetensors'}
try:
    GPU_VRAM_GIB=int(subprocess.check_output(['nvidia-smi','--query-gpu=memory.total','--format=csv,noheader,nounits'],text=True).split()[0])/1024
except Exception:
    GPU_VRAM_GIB=0
_quant=globals().get('REF2VA_QUANT','auto')
if _quant=='auto':
    _quant='int8' if GPU_VRAM_GIB>=38 else 'w4a8'
    _other='w4a8' if _quant=='int8' else 'int8'
    if not (MODELS_ROOT/'diffusion_models'/REF2VA_FILES[_quant]).is_file() and (MODELS_ROOT/'diffusion_models'/REF2VA_FILES[_other]).is_file():
        print(f'⚠ auto 偏好 {_quant}，但 Drive 只有 {_other}，改用 {_other}',flush=True)
        _quant=_other
REF2VA_SELECTED=_quant
REF2VA_FILE=REF2VA_FILES[_quant]
if _quant=='int8' and 0<GPU_VRAM_GIB<38:
    print(f'注意：GPU 只有 {GPU_VRAM_GIB:.0f}GiB VRAM，Ref2VA INT8 可能 offload 到 RAM 而變慢；請確認是高 RAM 執行階段',flush=True)
print(f'Ref2VA 主模型：{_quant} → {REF2VA_FILE}（GPU VRAM {GPU_VRAM_GIB:.0f}GiB）',flush=True)
COMMON=['qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors','minimax_h3_video_vae_int8_convrot.safetensors','minimax_h3_audio_vae_fp32.safetensors','minimax_h3_fl2va_pruned_int8_convrot.safetensors',REF2VA_FILE,'minimax_h3_video_vae_fp16.safetensors','minimax_h3_turbo_v4_step600_ema.safetensors']

# JSON manifest is stored in Google Drive. It contains filenames, sizes, mtimes, header result,
# and optional completed SHA256; never assumes a hash was checked when it was not.
CACHE_PATH=DRIVE_ROOT/'configs/model_validation_cache_v1.json'
CACHE_PATH.parent.mkdir(parents=True,exist_ok=True)
try:
    cache=json.loads(CACHE_PATH.read_text(encoding='utf-8'))
    if cache.get('version')!=1: cache={'version':1,'files':{}}
except (OSError,ValueError,TypeError): cache={'version':1,'files':{}}
records=cache.setdefault('files',{})

def save_cache():
    tmp=CACHE_PATH.with_suffix('.json.tmp')
    tmp.write_text(json.dumps(cache,ensure_ascii=False,indent=2),encoding='utf-8')
    os.replace(tmp,CACHE_PATH)

def valid_model(path):
    # Each run checks the safetensors header itself. No full tensor payload read.
    st=path.stat()
    if st.st_size<16: return False
    with path.open('rb') as f:
        first=f.read(8)
        if len(first)!=8: return False
        length=struct.unpack('<Q',first)[0]
        if not 2<=length<=min(100_000_000,st.st_size-8): return False
        header=json.loads(f.read(length))
        if not isinstance(header,dict): return False
        payload=st.st_size-8-length
        tensor_count=0
        for name,item in header.items():
            if name=='__metadata__': continue
            tensor_count+=1
            if not isinstance(item,dict):return False
            o=item.get('data_offsets')
            if not isinstance(o,list) or len(o)!=2 or not all(isinstance(x,int) for x in o):return False
            if not (0<=o[0]<=o[1]<=payload):return False
        return tensor_count>0

def scan_model(path,name,sha_expected=None):
    # Model identity: path, file size and nanosecond modification time.
    stat=path.stat()
    key=str(path)
    identity={'size':stat.st_size,'mtime_ns':stat.st_mtime_ns}
    past=records.get(key,{})
    if not valid_model(path):
        records.pop(key,None)
        return False
    item={'size':stat.st_size,'mtime_ns':stat.st_mtime_ns,'header_valid':True}
    if past.get('size')==stat.st_size and past.get('mtime_ns')==stat.st_mtime_ns and past.get('sha256'):
        item['sha256']=past['sha256']
    if VERIFY_MODEL_SHA256:
        if 'sha256' not in item:
            print('SHA256 完整核對:',name,flush=True)
            h=hashlib.sha256()
            with path.open('rb') as f:
                for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
            item['sha256']=h.hexdigest()
        if sha_expected and item['sha256'].lower()!=sha_expected.lower():
            records.pop(key,None)
            raise ValueError('SHA256 不符：'+str(path))
        if not sha_expected: print('額外 LoRA SHA256:',name,item['sha256'])
    records[key]=item
    return True

# mk13: Singularity V2 (optional). Revisions fixed to the README sources checked on 2026-10-24.
import requests
SINGULARITY_MODEL='Minimax-h3_Singularity_ref2va_v2_pruned_int8.safetensors'
SINGULARITY_LORA='minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors'
SINGULARITY_FILES={
 SINGULARITY_MODEL:{'folder':'diffusion_models','url':'https://huggingface.co/WarmBloodAban/Minimax-h3_Singularity/resolve/1259cf16cd0b655ba691a32fb46a409142762972/Minimax-h3_Singularity_ref2va_v2_pruned_int8.safetensors','sha256':'d7637c2c35cd30facd9e93778c491f6d82f91c4a441aecbdb4c384a62a5f3d9f','gib':19.6},
 SINGULARITY_LORA:{'folder':'loras','url':'https://huggingface.co/lightx2v/Minimax-h3-Turbo/resolve/30f485ed874aa44c8183348ad361c5eabad318da/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors','sha256':'5b9ab5ade15d0775676d01a907268a69a1468dc6033b3b0d3ded5502f3ebb84c','gib':1.9}}
CATALOG.update(SINGULARITY_FILES)

def download_file(url,target,sha256=None,attempts=4):
    """Stream into <target>.part while hashing; resume with HTTP Range after a drop; rename only when complete and verified."""
    target=Path(target); target.parent.mkdir(parents=True,exist_ok=True)
    part=target.with_name(target.name+'.part')
    for attempt in range(1,attempts+1):
        try:
            h=hashlib.sha256(); done=0
            if part.exists():
                with part.open('rb') as f:
                    for chunk in iter(lambda:f.read(16*1024*1024),b''): h.update(chunk); done+=len(chunk)
            headers={'Range':f'bytes={done}-'} if done else {}
            with requests.get(url,headers=headers,stream=True,timeout=(30,300),allow_redirects=True) as r:
                if done and r.status_code==200:
                    h=hashlib.sha256(); done=0; part.unlink()  # server ignored Range: start over
                if not (r.status_code in (200,206) or (done and r.status_code==416)):
                    raise OSError(f'HTTP {r.status_code}')
                if r.status_code!=416:  # 416: .part already holds the whole file
                    total=done+int(r.headers.get('content-length',0) or 0); mark=done
                    with part.open('ab') as f:
                        for chunk in r.iter_content(16*1024*1024):
                            f.write(chunk); h.update(chunk); done+=len(chunk)
                            if done-mark>=1024**3:
                                mark=done; print(f'  {target.name}: {done/1024**3:.1f}/{total/1024**3:.1f} GiB',flush=True)
            digest=h.hexdigest()
            if sha256 and digest.lower()!=sha256.lower():
                part.unlink(missing_ok=True)
                raise ValueError(f'SHA256 不符，已刪除下載檔：{target.name}')
            os.replace(part,target)
            print('下載完成:',target,flush=True)
            return digest
        except ValueError:
            raise
        except Exception as exc:
            print(f'  下載中斷（第 {attempt}/{attempts} 次），稍後續傳：',type(exc).__name__,str(exc)[:200],flush=True)
            if attempt==attempts: raise
            time.sleep(10*attempt)

def ensure_drive_space(gib,label):
    try: free=shutil.disk_usage(DRIVE_ROOT).free/1024**3
    except OSError: return
    assert free>gib+1, f'Drive 剩餘約 {free:.1f} GiB，不足以下載 {label}（約 {gib:.1f} GiB）；請清出空間，或在第 1 格關閉後重跑'

# mk21: every catalog model missing from Drive is downloaded into Drive first (SHA256 verified, resumable).
# Faster/cheaper on a CPU runtime with cell 0; this is the safety net on the GPU runtime.
if globals().get('AUTO_DOWNLOAD_MODELS',True):
    pending=[]
    for n,i in CATALOG.items():
        if (MODELS_ROOT/i['folder']/n).is_file(): continue
        if n in SINGULARITY_FILES and not globals().get('INSTALL_SINGULARITY'): continue
        if n=='REDQW21_V2_int8.safetensors': continue  # 不自動下載；有手動放就照常啟用
        pending.append((n,i))
    if pending: print(f'Drive 缺少 {len(pending)} 個模型，開始下載（只需一次；建議改用第 0 格在 CPU 執行階段先下載）',flush=True)
    for n,i in pending:
        target=MODELS_ROOT/i['folder']/n
        print('下載:',n,flush=True)
        try:
            digest=download_file(i['url'],target,i['sha256'])
        except Exception as exc:
            print('  ✗ 下載失敗：',type(exc).__name__,str(exc)[:200],flush=True)
            continue
        st=target.stat()
        records[str(target)]={'size':st.st_size,'mtime_ns':st.st_mtime_ns,'header_valid':True,'sha256':digest}
        save_cache()

found={}; invalid=[]
print('檔案檢查：只讀檔案大小與 safetensors 標頭；完整 SHA256:',VERIFY_MODEL_SHA256,flush=True)
for name,item in CATALOG.items():
    expected=MODELS_ROOT/item['folder']/name
    if expected.is_file():
        try: valid=scan_model(expected,name,item.get('sha256'))
        except (OSError,ValueError,TypeError) as exc:
            valid=False; print('檢查失敗:',name,str(exc))
        if valid:
            found[name]=expected
            print('OK',expected.relative_to(DRIVE_ROOT),f'({expected.stat().st_size/1024**3:.2f} GiB)',flush=True)
            continue
        invalid.append(str(expected))
    if name in REF2VA_FILES.values() and name not in COMMON:
        print('（未選用的 Ref2VA 量化版，可不放）',expected.relative_to(DRIVE_ROOT),flush=True)
        continue
    print('必要模型缺少/異常' if name in COMMON else '選用模型缺少/異常',expected.relative_to(DRIVE_ROOT),flush=True)
    print('  模型來源:',item['url'])
    # Only search misplaced files when necessary; recursive Drive scans can be costly.
for extra in (MODELS_ROOT/'loras').rglob('*.safetensors'):
    name=extra.relative_to(MODELS_ROOT/'loras').as_posix()
    if name in CATALOG:continue
    try: valid=scan_model(extra,name)
    except (OSError,ValueError,TypeError):valid=False
    if valid:
        CATALOG[name]={'folder':'loras','sha256':None,'url':'user supplied'}
        found[name]=extra
        print('EXTRA LORA',extra.relative_to(DRIVE_ROOT),flush=True)
    else:print('INVALID EXTRA LORA',name)
save_cache()
missing=[n for n in COMMON if n not in found]
assert not missing, 'H3 必要模型缺少或損壞，請依上列位置補齊後重跑本格：'+', '.join(missing)
qwen_shared=['qwen3vl_8b_int8_convrot.safetensors','qwen_image_2.1_vae_bf16.safetensors']
QWEN_AVAILABLE={key:name for key,name in {'original':'qwen_image_2.1_int8_convrot.safetensors','noct-q-v4':'NoctQ_V4_int8_convrot.safetensors','redqw21-v2':'REDQW21_V2_int8.safetensors'}.items() if name in found and all(x in found for x in qwen_shared)}
SWAP_AVAILABLE={key:name for key,name in {'head':'minimax_h3_head_swap_v1.0_r32.safetensors','character':'minimax_h3_character_swap_v1.0_ref2va_r64_gl.safetensors','akatz-v1.5':'h3_character_swap_v1.5.safetensors'}.items() if name in found}
HAS_SEED=all(n in found for n in ['seedvr2_3b_int8_convrot.safetensors','seedvr2_ema_vae_fp16.safetensors'])
HAS_CAMERA=bool(QWEN_AVAILABLE) and 'qwen_image_2.1_multiple_angles_v2_step1500.safetensors' in found
selected=set(COMMON)
if QWEN_AVAILABLE: selected.update(qwen_shared); selected.update(QWEN_AVAILABLE.values())
if HAS_CAMERA: selected.add('qwen_image_2.1_multiple_angles_v2_step1500.safetensors')
selected.update(SWAP_AVAILABLE.values())
selected.update(name for name in found if CATALOG[name]['folder']=='loras')
if HAS_SEED: selected.update(['seedvr2_3b_int8_convrot.safetensors','seedvr2_ema_vae_fp16.safetensors'])
HAS_SINGULARITY=bool(globals().get('INSTALL_SINGULARITY')) and all(n in found for n in SINGULARITY_FILES)
if HAS_SINGULARITY: selected.update(SINGULARITY_FILES)
ACTIVE_MODELS=ROOT/'h3_models' if COPY_MODELS_TO_SSD else MODELS_ROOT
if COPY_MODELS_TO_SSD:
    needed=sum(found[n].stat().st_size for n in selected if not (ACTIVE_MODELS/CATALOG[n]['folder']/n).exists())
    free=shutil.disk_usage(ROOT).free
    assert free>needed+8*1024**3, f'SSD 空間不足：本次複製需 {needed/1024**3:.1f}GiB，請清理或設定 COPY_MODELS_TO_SSD=False'
for name in sorted(selected):
    source=found[name]; target=ACTIVE_MODELS/CATALOG[name]['folder']/name
    if COPY_MODELS_TO_SSD:
        target.parent.mkdir(parents=True,exist_ok=True)
        if not target.exists() or target.stat().st_size!=source.stat().st_size or target.stat().st_mtime_ns!=source.stat().st_mtime_ns:
            part=target.with_suffix(target.suffix+'.part')
            print('複製到 SSD:',name,flush=True); shutil.copy2(source,part); os.replace(part,target)
cfg={'h3_colab':{'base_path':str(ACTIVE_MODELS)}}
for folder in ['diffusion_models','text_encoders','vae','loras']:
    if (ACTIVE_MODELS/folder).is_dir(): cfg['h3_colab'][folder]=folder
# JSON is valid YAML; avoids importing packages into the notebook kernel.
(COMFY/'extra_model_paths.yaml').write_text(json.dumps(cfg,indent=2),encoding='utf-8')
print('Ref2VA:',REF2VA_SELECTED,'可用 Qwen:',list(QWEN_AVAILABLE),'換角:',list(SWAP_AVAILABLE),'SeedVR2:',HAS_SEED,'攝影機:',HAS_CAMERA,'Singularity V2:',HAS_SINGULARITY)

# mk17: lazy SSD cache. A tiny custom node wraps folder_paths.get_full_path: while ComfyUI executes a workflow, a model
# resolved from Drive is first copied to /content/h3_models (same layout), then the SSD path is returned. Readiness/status
# checks (outside workflow execution) never trigger a copy.
SSD_CACHE_NODE=r"""
import logging, os, shutil, sys, threading, time
from pathlib import Path
import folder_paths

DRIVE = Path(__DRIVE__)
CACHE = Path(__CACHE__)
CACHE.mkdir(parents=True, exist_ok=True)
EXECUTION = os.path.normcase(os.path.join(os.path.dirname(os.path.abspath(folder_paths.__file__)), "execution.py"))
_lock = threading.Lock()
_original = folder_paths.get_full_path


def _executing():
    frame = sys._getframe(2)
    while frame is not None:
        if os.path.normcase(os.path.abspath(frame.f_code.co_filename)) == EXECUTION:
            return True
        frame = frame.f_back
    return False


def _cached(path):
    source = Path(path)
    try:
        target = CACHE / source.relative_to(DRIVE)
    except ValueError:
        return path
    size = source.stat().st_size
    if target.is_file() and target.stat().st_size == size:
        return str(target)
    if not _executing():
        return path
    with _lock:
        if target.is_file() and target.stat().st_size == size:
            return str(target)
        if shutil.disk_usage(CACHE).free < size + 8 * 1024**3:
            logging.warning("[H3 SSD cache] SSD space low, loading from Drive: %s", source.name)
            return path
        target.parent.mkdir(parents=True, exist_ok=True)
        part = target.with_name(target.name + ".part")
        started = time.monotonic()
        logging.info("[H3 SSD cache] copying %s (%.1f GiB) to SSD...", source.name, size / 1024**3)
        try:
            copied = mark = 0
            with source.open("rb") as src, part.open("wb") as dst:
                while chunk := src.read(64 * 1024 * 1024):
                    dst.write(chunk)
                    copied += len(chunk)
                    if copied - mark >= 2 * 1024**3:
                        mark = copied
                        logging.info("[H3 SSD cache] %s: %.1f/%.1f GiB", source.name, copied / 1024**3, size / 1024**3)
            if part.stat().st_size != size:
                raise OSError("size mismatch")
            os.replace(part, target)
        except Exception as error:
            part.unlink(missing_ok=True)
            logging.warning("[H3 SSD cache] copy failed, loading from Drive: %s (%s)", source.name, error)
            return path
        logging.info("[H3 SSD cache] %s ready on SSD in %d s", source.name, time.monotonic() - started)
        return str(target)


def get_full_path(folder_name, filename):
    path = _original(folder_name, filename)
    return _cached(path) if path else path


folder_paths.get_full_path = get_full_path
NODE_CLASS_MAPPINGS = {}
"""
cache_node=COMFY/'custom_nodes/h3_colab_ssd_cache'
if globals().get('LAZY_SSD_CACHE') and not COPY_MODELS_TO_SSD:
    cache_node.mkdir(parents=True,exist_ok=True)
    (cache_node/'__init__.py').write_text(SSD_CACHE_NODE.replace('__DRIVE__',repr(str(MODELS_ROOT))).replace('__CACHE__',repr(str(ROOT/'h3_models'))),encoding='utf-8')
    print('用到才複製到 SSD：開啟（快取位置 /content/h3_models，進度看 h3_comfy.log）')
elif cache_node.exists():
    shutil.rmtree(cache_node)


注意：GPU 只有 22GiB VRAM，Ref2VA INT8 可能 offload 到 RAM 而變慢；請確認是高 RAM 執行階段
Ref2VA 主模型：int8 → minimax_h3_ref2va_pruned_int8_convrot.safetensors（GPU VRAM 22GiB）
檔案檢查：只讀檔案大小與 safetensors 標頭；完整 SHA256: False
OK models/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors (14.61 GiB)
OK models/vae/minimax_h3_video_vae_int8_convrot.safetensors (2.95 GiB)
OK models/vae/minimax_h3_audio_vae_fp32.safetensors (0.56 GiB)
OK models/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors (19.53 GiB)
OK models/diffusion_models/minimax_h3_ref2va_pruned_w4a8_mixed.safetensors (10.96 GiB)
OK models/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors (19.53 GiB)
OK models/vae/minimax_h3_video_vae_fp16.safetensors (4.85 GiB)
OK models/loras/minimax_h3_turbo_v4_step600_ema.safetensors (0.73 GiB)
OK models/diffusion_models/qwen_image_2.1_int8_convrot.safetensors (6.76 GiB)
OK models/text_encoders/qwen3vl_8b_int8_convrot.safetensors (8.71 GiB)
OK models/vae/qwen_image_2.1_vae_bf16.sa

In [7]:
# 6. 修臉：與本機相同節點版本，Linux 安裝；僅規則／手動追蹤，不需 InsightFace。
if INSTALL_FACE_REFINE:
    checkout('https://github.com/Carasibana/ComfyUI-H3-FaceRefine.git',COMFY/'custom_nodes/ComfyUI-H3-FaceRefine','d8521d14fe0d721d80cd9417fff5a559cbc21aba')
    audio=COMFY/'custom_nodes/ComfyUI-H3-NativeAudioLock'
    audio.mkdir(parents=True,exist_ok=True)
    url='https://raw.githubusercontent.com/Shrek3OnVH5/MiniMax-H3-NativeAudio-MusicVideo-Workflow/84895cfaaef39abb42fb844b02cb42f868c7bebf/custom_nodes/ComfyUI-H3-NativeAudioLock/__init__.py'
    run(['curl','--fail','--location','--retry','3',url,'-o',audio/'__init__.py'])
    run([PYTHON,'-m','pip','uninstall','-y','opencv-python-headless'])
    run([PYTHON,'-m','pip','install','opencv-python==5.0.0.93'])
    run([PYTHON,'-m','pip','install','--no-deps','ultralytics==8.4.178','ultralytics-thop==2.2.2','ultralytics-platform==0.1.90','scenedetect==0.7.2','polars==2.0.0','polars-runtime-32==2.0.0','platformdirs==4.13.3','nvidia-ml-py==13.615.71'])
    detector=MODELS_ROOT/'ultralytics/bbox/face_yolov8m.pt'
    detector.parent.mkdir(parents=True,exist_ok=True)
    expected='717923c19b3f4bbf5250b728f1fa6b2cb72a33aed1d236ea9caf0e21ad943e5f'
    if not detector.exists():
        part=detector.with_suffix('.pt.part')
        run(['curl','--fail','--location','--retry','3','https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt','-o',part])
        assert hashlib.sha256(part.read_bytes()).hexdigest()==expected, '偵測器雜湊不符'
        os.replace(part,detector)
    assert hashlib.sha256(detector.read_bytes()).hexdigest()==expected, '既有偵測器雜湊不符'
    local=COMFY/'models/ultralytics/bbox/face_yolov8m.pt'; local.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(detector,local)
    run([PYTHON,'-c','from ultralytics import YOLO; import scenedetect; print("FaceRefine dependencies ready")'])
# FlashVSR (optional second 畫質提升 engine). Model lives in Drive (MyDrive/H3_Colab/models/FlashVSR-v1.1) and is
# symlinked into ComfyUI; if files are missing it downloads them into Drive once. Failure only disables the option.
HAS_FLASHVSR=False
if INSTALL_FLASHVSR:
    try:
        checkout('https://github.com/lihaoyun6/ComfyUI-FlashVSR_Ultra_Fast.git',COMFY/'custom_nodes/ComfyUI-FlashVSR_Ultra_Fast','4820b3f02347bddcbbb9a5a85ab7638fe976366e')
        run([PYTHON,'-m','pip','install','-q','einops','huggingface_hub','safetensors','tqdm'])
        need=['LQ_proj_in.ckpt','TCDecoder.ckpt','diffusion_pytorch_model_streaming_dmd.safetensors','Wan2.1_VAE.pth']
        drive_fv=MODELS_ROOT/'FlashVSR-v1.1'; drive_fv.mkdir(parents=True,exist_ok=True)
        if not all((drive_fv/n).is_file() for n in need):
            print('Drive 缺少 FlashVSR 模型，下載到',drive_fv,'（只需一次）…',flush=True)
            run([PYTHON,'-c','import sys;from huggingface_hub import snapshot_download;snapshot_download(repo_id="JunhaoZhuang/FlashVSR-v1.1",local_dir=sys.argv[1])',drive_fv])
        missing=[n for n in need if not (drive_fv/n).is_file()]
        assert not missing, 'FlashVSR 模型缺檔：'+', '.join(missing)
        fv_dir=COMFY/'models/FlashVSR-v1.1'
        if fv_dir.is_symlink(): fv_dir.unlink()
        elif fv_dir.exists(): shutil.rmtree(fv_dir)
        fv_dir.symlink_to(drive_fv,target_is_directory=True)
        HAS_FLASHVSR=True
        print('FlashVSR 就緒（模型來自 Drive）',flush=True)
    except Exception as exc:
        print('⚠ FlashVSR 安裝失敗，畫質提升只提供 SeedVR2：',type(exc).__name__,str(exc)[:300],flush=True)
# mk13: VOSR 1x refinement. Node pinned to the locally verified commit; models live in Drive and are symlinked into ComfyUI.
HAS_VOSR=False
if INSTALL_VOSR:
    try:
        checkout('https://github.com/ylchen333/ComfyUI-VOSR2.git',COMFY/'custom_nodes/ComfyUI-VOSR2','6a21810b3f4f1a0ffc477c55330db0ac03c671f0')
        run([PYTHON,'-m','pip','install','-q','-r',COMFY/'custom_nodes/ComfyUI-VOSR2/requirements.txt'])
        hf='https://huggingface.co/CSWRY/VOSR/resolve/f24b3061b7f350b81e1907bdcfc27f71fb4ff3f3/'
        bundle=MODELS_ROOT/'vosr2/VOSR2'
        vosr_files=[('VOSR2/args.json','args.json',None,0.0),
                    ('VOSR2/checkpoints/ema_model.safetensors','checkpoints/ema_model.safetensors','bdcaa81e4c675b6074de643e27daafe73eb8125d4d0264d1bf30a004e9644b71',5.2),
                    ('Qwen-Image-vae-2d/config.json','Qwen-Image-vae-2d/config.json',None,0.0),
                    ('Qwen-Image-vae-2d/diffusion_pytorch_model.safetensors','Qwen-Image-vae-2d/diffusion_pytorch_model.safetensors','1dd2c67f3f29734d48709513ed181d4f351966d76ba95f77a5cff8e1daf547fa',0.2)]
        dino=bundle/'dinov2_vitl14.safetensors'
        need=sum(g for _,rel,_,g in vosr_files if not (bundle/rel).is_file())+(1.2 if not dino.is_file() else 0)
        if need: ensure_drive_space(need,'VOSR')
        for remote,rel,sha,_ in vosr_files:
            if not (bundle/rel).is_file():
                print('Drive 缺少 VOSR 檔案，開始下載（只需一次）:',rel,flush=True)
                download_file(hf+remote,bundle/rel,sha)
        json.loads((bundle/'args.json').read_text(encoding='utf-8'))
        if not dino.is_file():
            # Upstream ships a torch pickle; convert on the local SSD, then move the safetensors into Drive.
            pth=ROOT/'h3_vosr_tmp/dinov2_vitl14_pretrain.pth'
            download_file(hf+'torch_cache/checkpoints/dinov2_vitl14_pretrain.pth',pth,'d5383ea8f4877b2472eb973e0fd72d557c7da5d3611bd527ceeb1d7162cbf428')
            local=pth.with_name('dinov2_vitl14.safetensors')
            run([PYTHON,'-c','import sys,torch\nfrom safetensors.torch import save_file\nstate=torch.load(sys.argv[1],map_location="cpu",weights_only=True,mmap=True)\nsave_file({k:v.contiguous() for k,v in state.items()},sys.argv[2])',pth,local])
            part=dino.with_name(dino.name+'.part'); shutil.copyfile(local,part); os.replace(part,dino)
            shutil.rmtree(pth.parent)
            print('DINOv2 已轉換並存入 Drive:',dino,flush=True)
        link=COMFY/'models/vosr2'
        if link.is_symlink(): link.unlink()
        elif link.exists():
            # ComfyUI-VOSR2 creates an empty models/vosr2 on import; never delete real files.
            assert not any(p.is_file() for p in link.rglob('*')), f'{link} 內已有檔案，請先手動移到 Drive 的 models/vosr2'
            shutil.rmtree(link)
        link.symlink_to(MODELS_ROOT/'vosr2',target_is_directory=True)
        HAS_VOSR=True
        print('VOSR 就緒（模型來自 Drive）',flush=True)
    except Exception as exc:
        print('⚠ VOSR 安裝失敗，畫質提升不提供 VOSR：',type(exc).__name__,str(exc)[:300],flush=True)
# ultralytics-platform 0.1.90 requires httpx (confirmed missing on Colab).
run([PYTHON,'-m','pip','install','httpx'])
run([PYTHON,'-m','pip','check'])


FlashVSR 就緒（模型來自 Drive）
VOSR 就緒（模型來自 Drive）


In [8]:
# 7. 只套用到 Colab checkout；固定版本與精確匹配避免改錯位置。
import re
def pristine(rel):
    return subprocess.check_output(['git','-C',REPO,'show',STUDIO_COMMIT+':'+rel],text=True,encoding='utf-8')
def replace(text,old,new,count=1):
    assert text.count(old)==count, f'Colab patch mismatch: {old[:100]} ({text.count(old)} != {count})'
    return text.replace(old,new)
def save(rel,text):
    (REPO/rel).write_text(text,encoding='utf-8')
for rel in ('lib/comfy.ts','lib/tts.ts','lib/drama3.ts'):
    path='h3-local-studio/'+rel
    s=replace(pristine(path),'http://127.0.0.1:8188','/comfy')
    if rel=='lib/comfy.ts':
        # Ref2VA main model follows REF2VA_QUANT (cell 1/5). Fail loudly if the Studio source changed.
        _w4='minimax_h3_ref2va_pruned_w4a8_mixed.safetensors'
        assert s.count(_w4)>=1, 'lib/comfy.ts 找不到 Ref2VA W4A8 檔名，Studio 原始碼已變更，請檢查 Ref2VA 載入段落'
        if REF2VA_FILE!=_w4: s=s.replace(_w4,REF2VA_FILE)
        # Colab: datacenter GPU needs no per-step cooldown. Default 0, minimum 0.
        s=replace(s,'export const MIN_COOLDOWN_SECONDS = 5;','export const MIN_COOLDOWN_SECONDS = 0;')
        s=replace(s,'export const DEFAULT_COOLDOWN_SECONDS = 15;','export const DEFAULT_COOLDOWN_SECONDS = 0;')
        # A finished ComfyUI result must win over a late/stray stop flag ("工作已停止" after a completed face job).
        s=replace(s,'    if (stopped?.()) throw new Error("工作已停止。");\n','')
        s=replace(s,'    const queueResponse = await request("/queue");','    if (stopped?.()) throw new Error("工作已停止。");\n    const queueResponse = await request("/queue");')
        # Colab tunnel: transient network drops abort whole jobs ("Failed to fetch"). Retry idempotent GETs.
        s=replace(s,'  const response = await fetch(`${COMFY_URL}${path}`, init);','''  const method = (init?.method ?? "GET").toUpperCase();
  let response: Response;
  for (let attempt = 0; ; attempt++) {
    try {
      response = await fetch(`${COMFY_URL}${path}`, init);
      if (method === "GET" && [502, 503, 504, 520, 522, 523, 524, 530].includes(response.status) && attempt < 40) throw new Error(`tunnel ${response.status}`);
      break;
    } catch (error) {
      if (method !== "GET" || attempt >= 40) throw error;
      await new Promise((resolve) => setTimeout(resolve, 3000));
    }
  }''')
    save(path,s)
save('h3-local-studio/lib/vosr.ts',replace(pristine('h3-local-studio/lib/vosr.ts'),'http://127.0.0.1:8188/view','/comfy/view'))
page=pristine('h3-local-studio/app/page.tsx')
if not QWEN_AVAILABLE: page=replace(page,'    { id: "edit", icon: "✎", label: "圖像編輯" },\n','')
if not GEMINI_KEY: page=replace(page,'    { id: "tts", icon: "♫", label: "語音生成" },\n','')
page=replace(page,'disabled={!video.extendable}','disabled={true} title="Colab 停用延伸"')
page=replace(page,'disabled={deleting === key}','disabled={true} title="Colab 停用刪除"',2)
page=replace(page,'disabled={deleting === `${audio.subfolder}/${audio.filename}`}','disabled={true} title="Colab 停用刪除"')
page=replace(page,'{video.extendable && <small>','{false && <small>')
# Remove mode tabs that cannot run with this runtime/model set.
lines=page.splitlines(keepends=True)
labels=['RTX提升']+([] if SWAP_AVAILABLE else ['影片換角'])+([] if HAS_SEED or HAS_VOSR else ['影片畫質提升'])
for label in labels:
    matches=[line for line in lines if line.strip().startswith('<button ') and '>'+label+'</button>' in line]
    assert len(matches)==1, label
    lines.remove(matches[0])
page=''.join(lines)
page=replace(page,'disabled={!connected || isOptimizing || Boolean(continuationSource)}','disabled={true} title="Colab 未配置 Codex／Grok 優化服務"')
page=replace(page,'disabled={!connected || isOptimizing}','disabled={true} title="Colab 未配置 Codex 優化服務"')
if not INSTALL_FACE_REFINE:
    page=replace(page,'checked={faceEnabled} onChange=', 'checked={faceEnabled} disabled title="未安裝修臉功能" onChange=')
    page=replace(page,'<button role="menuitem" onClick={() => { setOpenMenu(null); onRefine(video); }}>','<button disabled title="未安裝修臉功能" role="menuitem" onClick={() => { setOpenMenu(null); onRefine(video); }}>')
if not HAS_CAMERA:
    page=replace(page,'checked={qwenCameraEnabled} onChange=', 'checked={qwenCameraEnabled} title="缺少攝影機 LoRA" disabled={true} onChange=')
    # Avoid duplicate disabled attributes on the same input.
    page=page.replace('setQwenMention(null); }} disabled={isOptimizing} />','setQwenMention(null); }} />')
if QWEN_AVAILABLE:
    default=next(iter(QWEN_AVAILABLE))
    page=replace(page,'useState<QwenModel>("original")','useState<QwenModel>('+json.dumps(default)+')')
    page=replace(page,'Object.entries(QWEN_MODELS).map','Object.entries(QWEN_MODELS).filter(([key]) => '+json.dumps(list(QWEN_AVAILABLE))+'.includes(key)).map')
page=replace(page,'文字、首尾幀或多圖參考，在你的 RTX 4080 Laptop 上生成有聲影片。','文字、首尾幀或多圖參考，在 Colab GPU 生成影片，完成後自動備份到 Google Drive。')
# Colab: offer 768P (native H3, aspect-scaled) for text and first/last-frame modes too.
page=replace(page,'                          <option value="p540">{sizeLabels.p540} · 540P</option>\n                        </>\n                      )}',
             '                          <option value="p540">{sizeLabels.p540} · 540P</option>\n                          <option value="native">{sizeLabels.native} · 768P</option>\n                        </>\n                      )}')
# Face-refine stop buttons: confirm first, and log a stack trace so a stray trigger is visible in DevTools.
page=replace(page,'onClick={() => void cancelFace(activeFaceControl).catch((caught) => setError(String(caught)))}','onClick={() => { if (!window.confirm("確定要停止人臉精修嗎？原片會保留。")) return; console.warn("[H3] face stop clicked", new Error().stack); void cancelFace(activeFaceControl).catch((caught) => setError(String(caught))); }}')
page=replace(page,'畫質提升 · 720p · {video.enhancementStrength}%','畫質提升 · {Math.min(video.width ?? 0, video.height ?? 0) || 720}p · {video.enhancementStrength}%')
if not HAS_SINGULARITY:
    page=replace(page,'                      <option value="singularity-v2">Singularity V2 · INT8</option>\n','')
if not HAS_VOSR:
    page=replace(page,'<button role="menuitem" onClick={() => { setOpenMenu(null); onVOSR(video); }}>VOSR 精修</button>','<button disabled title="VOSR 未安裝" role="menuitem" onClick={() => { setOpenMenu(null); onVOSR(video); }}>VOSR 精修</button>')
# mk14: per-mode defaults (text: Singularity V2 when installed; image/reference: original H3).
TEXT_MODEL='singularity-v2' if HAS_SINGULARITY else 'original'
page=replace(page,'useState<H3Model>("original")','useState<H3Model>('+json.dumps(TEXT_MODEL)+')')
page=replace(page,'useState<GenerationOptions["profile"]>("fast")','useState<GenerationOptions["profile"]>("cooled-turbo-8")')
page=replace(page,'useState<H3AttentionMethod>("sage")','useState<H3AttentionMethod>("sol")')
page=replace(page,'useState<GenerationOptions["resolution"]>("safe")','useState<GenerationOptions["resolution"]>("p480")')
page=replace(page,'  const [duration, setDuration] = useState(5);\n','  const [duration, setDuration] = useState(10);\n')
page=replace(page,'  const [soundEnabled, setSoundEnabled] = useState(true);\n','  const [soundEnabled, setSoundEnabled] = useState(true);\n  const modeSettings = useRef<Partial<Record<SourceMode, { h3Model: H3Model; profile: GenerationOptions["profile"]; attentionMethod: H3AttentionMethod; resolution: GenerationOptions["resolution"]; duration: number; aspect: GenerationOptions["aspect"]; soundEnabled: boolean }>>>({});\n')
page=replace(page,"""    if (nextMode === "reference") {
      setProfile(h3Model === "singularity-v2" ? "cooled-turbo-4" : "cooled-turbo-8");
      setResolution("native");
      setAspect("16:9");
    } else {
      if (resolution === "native") setResolution("safe");
      if (aspect === "4:3" || aspect === "3:4") setAspect("16:9");
    }
""","""    // Colab: each mode keeps its own settings for this page; the first visit uses that mode's defaults.
    modeSettings.current[sourceMode] = { h3Model, profile, attentionMethod, resolution, duration, aspect, soundEnabled };
    const next = modeSettings.current[nextMode] ?? { h3Model: nextMode === "text" ? """+json.dumps(TEXT_MODEL)+""" as H3Model : "original" as H3Model, profile: "cooled-turbo-8" as const, attentionMethod: nextMode === "reference" ? "sage" as const : "sol" as const, resolution: "p480" as const, duration: 10, aspect: "16:9" as const, soundEnabled: true };
    setH3Model(next.h3Model);
    setProfile(next.profile);
    setAttentionMethod(next.attentionMethod);
    setResolution(next.resolution);
    setDuration(next.duration);
    setAspect(next.aspect);
    setSoundEnabled(next.soundEnabled);
""")
save('h3-local-studio/app/page.tsx',page)
# Once the face job has produced its result, finish compositing even if a stop flag was set late.
fr='h3-local-studio/lib/face-refine.ts'
s=pristine(fr)
s=replace(s,'    const raw = await sampleFace(source, input, control, onStatus, picks);\n    checkStop();\n','    const raw = await sampleFace(source, input, control, onStatus, picks);\n')
save(fr,s)
speech=pristine('h3-local-studio/app/speech-studio.tsx')
speech=replace(speech,'useState<"drama3" | "gemini">("drama3")','useState<"drama3" | "gemini">("gemini")')
speech=''.join(line for line in speech.splitlines(keepends=True) if not ('<button ' in line and 'setEngine("drama3")' in line) and '<div hidden={engine !== "drama3"}>' not in line)
save('h3-local-studio/app/speech-studio.tsx',speech)
if SWAP_AVAILABLE:
    swap=pristine('h3-local-studio/app/swap-studio.tsx')
    default=next(iter(SWAP_AVAILABLE))
    swap=replace(swap,'useState<SwapMode>("head")','useState<SwapMode>('+json.dumps(default)+')')
    swap=replace(swap,'useState(buildSwapPrompt("head"))','useState(buildSwapPrompt('+json.dumps(default)+'))')
    import re
    for mode in ('head','character','akatz-v1.5'):
        if mode not in SWAP_AVAILABLE:
            swap,n=re.subn(r'<option value="'+re.escape(mode)+r'">[^<]*</option>','',swap)
            assert n==1
    save('h3-local-studio/app/swap-studio.tsx',swap)
# Safe-long's CPU upscale no longer invokes the Windows Recycle Bin.
# Keep the source video; disabled deletion/continuation endpoints are blocked by nginx below.
api=pristine('integrations/ComfyUI-H3-Studio/h3_studio_api.py')
api=replace(api,'    recycle_file(source)\n','    # Colab: retain the local source; no Windows Recycle Bin call.\n')
# mk22: last ComfyUI log lines for the queue popup (progress bars collapsed to their latest update).
H3_LOG_ROUTE=r'''H3_COMFY_LOG = Path(__H3_LOG__)
_H3_ANSI = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]")
_H3_PROGRESS = re.compile(r"\d+%\||\d+/\d+ \[")


@PromptServer.instance.routes.get("/h3-studio/log-tail")
async def h3_log_tail(request):
    try:
        count = max(1, min(40, int(request.query.get("lines", "10"))))
    except ValueError:
        count = 10
    try:
        with H3_COMFY_LOG.open("rb") as handle:
            handle.seek(0, os.SEEK_END)
            handle.seek(max(0, handle.tell() - 65536))
            text = handle.read().decode("utf-8", errors="replace")
    except OSError:
        return web.json_response({"lines": []}, headers={"Cache-Control": "no-store"})
    lines = []
    for raw in re.split(r"[\r\n]+", _H3_ANSI.sub("", text)):
        line = raw.rstrip()
        if not line.strip():
            continue
        if lines and _H3_PROGRESS.search(line) and _H3_PROGRESS.search(lines[-1]):
            lines[-1] = line[-240:]
        else:
            lines.append(line[-240:])
    return web.json_response({"lines": lines[-count:]}, headers={"Cache-Control": "no-store"})


@PromptServer.instance.routes.get("/h3-studio/outputs")
'''.replace('__H3_LOG__',repr(str(ROOT/'h3_comfy.log')))
api=replace(api,'@PromptServer.instance.routes.get("/h3-studio/outputs")\n',H3_LOG_ROUTE)
save('integrations/ComfyUI-H3-Studio/h3_studio_api.py',api)
# Enhancement: selectable 720p / 1080p / 1440p target (short side).
enh='integrations/ComfyUI-H3-Studio/h3_studio_enhance.py'
s=pristine(enh)
s=replace(s,'''def target_dimensions(width, height):
    if min(width, height) >= 720:
        raise ValueError("目前提供升至 720p，請選擇短邊小於 720 的影片。")
    scale = 720 / min(width, height)''','''def target_dimensions(width, height, target=720):
    if target not in (720, 1080, 1440):
        raise ValueError("目標解析度只支援 720p、1080p、1440p。")
    if min(width, height) >= target:
        raise ValueError(f"來源短邊已達 {min(width, height)}，請選擇高於來源的目標解析度。")
    scale = target / min(width, height)''')
s=replace(s,'''        width, height = target_dimensions(info["width"], info["height"])
        if max(width, height) > 2160:
            raise ValueError("影片比例過長，720p 長邊上限為 2160。")''','''        target = int(options.get("target", 720))
        width, height = target_dimensions(info["width"], info["height"], target)
        if max(width, height) > target * 3:
            raise ValueError(f"影片比例過長，{target}p 長邊上限為 {target * 3}。")''')
save(enh,s)
lib='h3-local-studio/lib/enhance.ts'
s=pristine(lib)
s=replace(s,'export async function prepareEnhancement(source: File | GeneratedVideo, start: number, duration: number) {','export async function prepareEnhancement(source: File | GeneratedVideo, start: number, duration: number, target = 720) {')
s=replace(s,'    body.append("duration", String(duration));\n','    body.append("duration", String(duration));\n    body.append("target", String(target));\n')
s=replace(s,'JSON.stringify({ source, start, duration })','JSON.stringify({ source, start, duration, target })')
save(lib,s)
ui='h3-local-studio/app/enhance-studio.tsx'
s=pristine(ui)
s=replace(s,'  const [strength, setStrength] = useState(50);\n','  const [strength, setStrength] = useState(50);\n  const [target, setTarget] = useState(720);\n')
s=replace(s,'prepareEnhancement(source, segment ? start : 0, segment ? duration : 0)','prepareEnhancement(source, segment ? start : 0, segment ? duration : 0, target)')
s=replace(s,'<p>升至 720p，保留原比例','<p>升至所選解析度（短邊），保留原比例')
s=replace(s,'      <label className="enhancement-strength">','      <label><span>目標解析度</span><select value={target} disabled={locked} onChange={(event) => { reset(); setTarget(Number(event.target.value)); }}><option value={720}>720p</option><option value={1080}>1080p</option><option value={1440}>1440p</option></select></label>\n      <label className="enhancement-strength">')
s=replace(s,'<h3>畫質提升 · 720p · ','<h3>畫質提升 · {result.width} × {result.height} · ')
save(ui,s)
# Enhancement: no 15 s cap; duration auto-fills with the source video's full length.
s=(REPO/enh).read_text(encoding='utf-8')
s=replace(s,'''        if duration > 15.1:
            raise ValueError("第一版每次最多處理 15 秒，請改用片段測試。")
''','')
save(enh,s)
s=(REPO/ui).read_text(encoding='utf-8')
s=replace(s,'每次最多 15 秒；建議先測試 3 秒。','片段長度會自動帶入影片全長，可自行縮短先測試。')
s=replace(s,'aria-label="來源影片預覽" />','aria-label="來源影片預覽" onLoadedMetadata={(event) => { const total = event.currentTarget.duration; if (Number.isFinite(total) && total > 0) { setStart(0); setDuration(Math.floor(total * 10) / 10); } }} />')
s=replace(s,'min="0.1" max="15" step="0.1"','min="0.1" step="0.1"')
s=replace(s,' || duration > 15))}','))}')
save(ui,s)
# Enhancement joins the shared generation queue (no screen lock; can queue behind other jobs / several targets).
s=(REPO/ui).read_text(encoding='utf-8')
s=replace(s,'import { useEffect, useState } from "react";','import { useEffect, useRef, useState } from "react";')
s=replace(s,'function SeedVRStudio({ videos, connected, disabled, executionDisabled, onBusyChange, onComplete }: {','function SeedVRStudio({ videos, connected, disabled, executionDisabled, onBusyChange, onComplete, enqueue }: {')
s=replace(s,'  onComplete: () => void;\n}) {','  onComplete: () => void;\n  enqueue: (label: string, run: (onStatus: (message: string) => void) => Promise<void>, onDiscard?: () => void) => number;\n}) {\n  void executionDisabled; void onBusyChange;\n  const selection = useRef(0);')
s=replace(s,'  function reset() { setCache(null);','  function reset() { selection.current += 1; setCache(null);')
start=s.index('  async function enhance() {'); end=s.index('  const locked = busy || disabled;')
s=s[:start]+'''  function enhance() {
    if (!source) return;
    const chosenStrength = strength, src = source, seg = segment, st = start, du = duration, tg = target, token = selection.current;
    const sameSelection = () => token === selection.current;
    const name = src instanceof File ? src.name : src.filename;
    setError(""); setStatus("已加入生成佇列，可切換到其他功能。");
    enqueue(`畫質提升 · ${name} · ${tg}p · ${chosenStrength}%`, async (onStatus) => {
      const report = (text: string) => { if (sameSelection()) setStatus(text); onStatus(text); };
      let current = sameSelection() ? cache : null;
      try {
        if (!current) {
          report("正在準備影片片段…");
          current = { input: await prepareEnhancement(src, seg ? st : 0, seg ? du : 0, tg), raw: null };
          if (sameSelection()) setCache(current);
        }
        if (chosenStrength > 0 && !current.raw) {
          current = { ...current, raw: await runEnhancement(current.input, report) };
          if (sameSelection()) setCache(current);
        }
        report("正在混合原片並另存成品…");
        const completed = await blendEnhancement(current.input, current.raw, chosenStrength);
        if (sameSelection()) setResult(completed);
        onComplete();
        report(`已完成 ${completed.width} × ${completed.height} · ${chosenStrength}% 強度`);
      } catch (caught) {
        if (sameSelection()) setError(caught instanceof Error ? caught.message : "影片畫質提升失敗。");
        throw caught;
      }
    }, () => { if (sameSelection()) setStatus("已從佇列移除畫質提升。"); });
  }

'''+s[end:]
s=replace(s,'disabled={!connected || locked || executionDisabled || !source ||','disabled={!connected || locked || !source ||')
s=replace(s,'{busy ? `正在處理 · ${elapsed} 秒` : cache?.raw ? "套用強度並另存" : "開始提升畫質"}','{cache?.raw ? "套用強度並另存（加入佇列）" : "加入畫質提升佇列"}')
s=replace(s,'      {executionDisabled && <p>可先設定，等待生成佇列完成後即可開始提升畫質。</p>}\n','')
save(ui,s)
qp='h3-local-studio/app/generation-queue-panel.tsx'
s=pristine(qp)
s=replace(s,'job.label.startsWith("RTX · ") ? "RTX 提升" :','job.label.startsWith("RTX · ") ? "RTX 提升" : job.label.startsWith("畫質提升 · ") ? "畫質提升" : job.label.startsWith("VOSR 1× 精修 · ") ? "VOSR 精修" :')
# mk22: while the popup of a running job is open, show the last 10 ComfyUI log lines (refresh every 2 s).
s=replace(s,'import { KeyboardEvent, useRef, useState } from "react";','import { KeyboardEvent, useEffect, useRef, useState } from "react";')
s=replace(s,'  const toggle = useRef<HTMLButtonElement>(null);\n',r'''  const toggle = useRef<HTMLButtonElement>(null);
  const [logLines, setLogLines] = useState<string[]>([]);
  const logJobRunning = popup !== null && jobs.some((job) => job.id === popup.id && job.status === "running");
  useEffect(() => {
    if (!logJobRunning) { setLogLines([]); return; }
    let alive = true;
    const load = async () => {
      try {
        const response = await fetch("/comfy/h3-studio/log-tail?lines=10", { cache: "no-store" });
        if (response.ok && alive) setLogLines(((await response.json()) as { lines?: string[] }).lines ?? []);
      } catch { /* tunnel hiccup: keep the previous lines */ }
    };
    void load();
    const timer = window.setInterval(() => void load(), 2000);
    return () => { alive = false; window.clearInterval(timer); };
  }, [logJobRunning]);
''')
s=replace(s,'''                  <p className="queue-message" role={job.status === "failed" ? "alert" : "status"}>{job.message}</p>
''',r'''                  <p className="queue-message" role={job.status === "failed" ? "alert" : "status"}>{job.message}</p>
                  {job.status === "running" && logLines.length > 0 && (
                    <pre aria-label="ComfyUI 日誌" style={{ margin: "8px 0 0", padding: "8px 10px", borderRadius: 8, background: "#0e0d13", color: "#b9b3c6", fontSize: 11, lineHeight: 1.5, whiteSpace: "pre-wrap", overflowWrap: "anywhere", maxHeight: 220, overflowY: "auto" }}>{logLines.join("\n")}</pre>
                  )}
''')
save(qp,s)
# Enhancement auto-chunking: long clips are split into short segments (RAM stays bounded), then joined.
s=(REPO/enh).read_text(encoding='utf-8')
s=replace(s,'import math\n','import math\nimport re\n')
s=replace(s,'''            options = await request.json()
            item = options.get("source", {})
            if not isinstance(item, dict):
                raise ValueError("無效的影片來源。")
            source = resolve_output_video(item.get("filename"), item.get("subfolder", ""))
            source_name = source.name
''','''            options = await request.json()
            if options.get("fromSourceId"):
                base_clip, base_manifest = input_paths(options["fromSourceId"])
                source = base_clip
                source_name = json.loads(base_manifest.read_text(encoding="utf-8")).get("sourceName", base_clip.name) if base_manifest.is_file() else base_clip.name
            else:
                item = options.get("source", {})
                if not isinstance(item, dict):
                    raise ValueError("無效的影片來源。")
                source = resolve_output_video(item.get("filename"), item.get("subfolder", ""))
                source_name = source.name
''')
s=replace(s,'def blend_video(source_id, raw, strength):','def blend_video(source_id, raw, strength, segment=False):')
s=replace(s,'    output = Path(folder_paths.get_output_directory()) / "video" / f"H3_Studio_Enhanced_{uuid.uuid4().hex[:12]}.mp4"\n',
            '    output = (clip.parent / f"{source_id}_seg_{uuid.uuid4().hex[:12]}.mp4") if segment else Path(folder_paths.get_output_directory()) / "video" / f"H3_Studio_Enhanced_{uuid.uuid4().hex[:12]}.mp4"\n')
s=replace(s,'        metadata = {"kind": "video", "enhanced": True, "enhancementStrength": strength,\n                    "enhancementSource": info["sourceName"], "width": width, "height": height,',
            '        if segment:\n            return {"segmentFile": output.name}\n        metadata = {"kind": "video", "enhanced": True, "enhancementStrength": strength,\n                    "enhancementSource": info["sourceName"], "width": width, "height": height,')
s=replace(s,'blend_video, data.get("sourceId"), data.get("raw"), data.get("strength"))','blend_video, data.get("sourceId"), data.get("raw"), data.get("strength"), bool(data.get("segment")))')
s+='''

SEGMENT_NAME = re.compile(r"^[0-9a-f]{32}_seg_[0-9a-f]{12}\\.mp4$")


def join_segments(names, strength):
    root = Path(folder_paths.get_input_directory()) / "h3_enhance"
    if not isinstance(names, list) or not 2 <= len(names) <= 1000 or not all(isinstance(n, str) and SEGMENT_NAME.match(n) for n in names):
        raise ValueError("無效的分段清單。")
    if isinstance(strength, bool) or not isinstance(strength, (int, float)) or not 0 <= strength <= 100:
        raise ValueError("強度必須介於 0～100%。")
    files = [root / n for n in names]
    if not all(f.is_file() for f in files):
        raise ValueError("分段檔案不存在，請重新處理。")
    manifest = root / f"{names[0][:32]}.json"
    info = json.loads(manifest.read_text(encoding="utf-8")) if manifest.is_file() else {}
    executable = shutil.which("ffmpeg")
    if not executable:
        raise ValueError("找不到 FFmpeg。")
    output = Path(folder_paths.get_output_directory()) / "video" / f"H3_Studio_Enhanced_{uuid.uuid4().hex[:12]}.mp4"
    output.parent.mkdir(parents=True, exist_ok=True)
    listing = root / f"join_{uuid.uuid4().hex[:12]}.txt"
    listing.write_text("".join(f"file '{f.as_posix()}'\\n" for f in files), encoding="utf-8")
    try:
        result = subprocess.run([executable, "-hide_banner", "-loglevel", "error", "-f", "concat", "-safe", "0", "-i", str(listing),
                                 "-map", "0:v:0", "-map", "0:a?", "-c:v", "libx264", "-crf", "18", "-preset", "fast",
                                 "-c:a", "aac", "-y", str(output)], capture_output=True, text=True, timeout=7200)
        if result.returncode:
            raise ValueError(result.stderr.strip()[-1200:] or "分段接合失敗。")
        final = probe_video(output)
        metadata = {"kind": "video", "enhanced": True, "enhancementStrength": strength,
                    "enhancementSource": info.get("sourceName", "影片"), "width": final["width"], "height": final["height"],
                    "duration": final["duration"], "sound": final["sound"], "extendable": False, "enhancementSegments": len(files)}
        metadata_path_for(output).write_text(json.dumps(metadata, ensure_ascii=False), encoding="utf-8")
        for f in files:
            f.unlink(missing_ok=True)
        return {"filename": output.name, "subfolder": "video", "type": "output",
                "modifiedAt": output.stat().st_mtime, **metadata}
    except Exception:
        output.unlink(missing_ok=True)
        raise
    finally:
        listing.unlink(missing_ok=True)


@PromptServer.instance.routes.post("/h3-studio/enhance/join")
async def join_output(request):
    try:
        data = await request.json()
        result = await asyncio.to_thread(join_segments, data.get("segments"), data.get("strength"))
        return web.json_response(result)
    except (ValueError, TypeError, KeyError, OSError, subprocess.SubprocessError) as error:
        return web.json_response({"error": str(error)}, status=400)
'''
save(enh,s)
s=(REPO/lib).read_text(encoding='utf-8')
s+='''
export async function prepareEnhancementFrom(sourceId: string, start: number, duration: number, target = 720) {
  return await (await request("/h3-studio/enhance/prepare", {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ fromSourceId: sourceId, start, duration, target }),
  })).json() as EnhancementInput;
}

export async function blendEnhancementSegment(source: EnhancementInput, raw: GeneratedVideo | null, strength: number) {
  return await (await request("/h3-studio/enhance/blend", {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ sourceId: source.sourceId, raw, strength, segment: true }),
  })).json() as { segmentFile: string };
}

export async function joinEnhancement(segments: string[], strength: number) {
  return await (await request("/h3-studio/enhance/join", {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ segments, strength }),
  })).json() as GeneratedVideo;
}
'''
save(lib,s)
s=(REPO/ui).read_text(encoding='utf-8')
s=replace(s,'import { blendEnhancement, EnhancementInput, prepareEnhancement, runEnhancement } from "@/lib/enhance";','import { blendEnhancement, blendEnhancementSegment, EnhancementInput, joinEnhancement, prepareEnhancement, prepareEnhancementFrom, runEnhancement } from "@/lib/enhance";')
s=replace(s,'  const [target, setTarget] = useState(720);\n','  const [target, setTarget] = useState(720);\n  const [sourceLength, setSourceLength] = useState(0);\n')
s=replace(s,'if (Number.isFinite(total) && total > 0) { setStart(0);','if (Number.isFinite(total) && total > 0) { setSourceLength(Math.floor(total * 10) / 10); setStart(0);')
s=replace(s,'      let current = sameSelection() ? cache : null;\n      try {','''      const total = seg ? du : (sourceLength || du);
      const offset = seg ? st : 0;
      const chunk = tg >= 1440 ? 5 : tg >= 1080 ? 10 : 20;
      if (total > chunk + 0.05) {
        try {
          report("正在準備完整片段…");
          const base = await prepareEnhancement(src, offset, total, tg);
          const pieces: [number, number][] = [];
          for (let t = 0; t < total - 0.05; t += chunk) pieces.push([t, t + chunk >= total - 0.05 ? 0 : chunk]);
          const segments: string[] = [];
          for (const [index, [at, length]] of pieces.entries()) {
            const tag = `第 ${index + 1}/${pieces.length} 段`;
            report(`${tag}：準備中…`);
            const input = await prepareEnhancementFrom(base.sourceId, at, length, tg);
            const raw = chosenStrength > 0 ? await runEnhancement(input, (message) => report(`${tag}：${message}`)) : null;
            report(`${tag}：混合原片…`);
            segments.push((await blendEnhancementSegment(input, raw, chosenStrength)).segmentFile);
          }
          report("正在接合所有分段並另存成品…");
          const completed = await joinEnhancement(segments, chosenStrength);
          if (sameSelection()) setResult(completed);
          onComplete();
          report(`已完成 ${completed.width} × ${completed.height} · ${pieces.length} 段 · ${chosenStrength}% 強度`);
        } catch (caught) {
          if (sameSelection()) setError(caught instanceof Error ? caught.message : "影片畫質提升失敗。");
          throw caught;
        }
        return;
      }
      let current = sameSelection() ? cache : null;
      try {''')
s=replace(s,'片段長度會自動帶入影片全長，可自行縮短先測試。','片段長度會自動帶入影片全長；長片會自動分段處理（720p 每段 20 秒、1080p 10 秒、1440p 5 秒）再接合。')
save(ui,s)
# LongMedia @96c50f7 wraps H3 FinalLayer.forward with the old 4-arg signature; ComfyUI v0.37.0
# passes 7 args (sigma, sample_sigmas, shifts) -> Ref2VA fails "_streamed_forward() takes 5 positional
# arguments but 8 were given". Delegate to the stock forward when the new args are present.
lm_dir=COMFY/'custom_nodes/ComfyUI-MiniMax-H3-LongMedia'
lm_src=subprocess.check_output(['git','-C',str(lm_dir),'show','96c50f7febc8c2e80c2a50d030d6cb3ff3de7ee3:nodes.py'],text=True,encoding='utf-8')
lm_src=replace(lm_src,'        def _streamed_forward(layer, x, t_emb, video_seg, audio_seg):\n',
    '        def _streamed_forward(layer, x, t_emb, video_seg, audio_seg, *extra):\n'
    '            if extra:\n'
    '                return original(x, t_emb, video_seg, audio_seg, *extra)\n')
(lm_dir/'nodes.py').write_text(lm_src,encoding='utf-8')
# Enhancement speed: VAE tiles / SeedVR2 temporal chunk scale with VRAM; RAM-aware segment length; join copies video.
s=(REPO/lib).read_text(encoding='utf-8')
s=replace(s,'export function buildEnhancementWorkflow(source: EnhancementInput) {','export type EnhancementTuning = { tile: number; temporal: number; chunkFrames: number };\nconst DEFAULT_TUNING: EnhancementTuning = { tile: 512, temporal: 16, chunkFrames: 9 };\n\nexport function buildEnhancementWorkflow(source: EnhancementInput, tuning: EnhancementTuning = DEFAULT_TUNING) {')
s=replace(s,'"7": node("VAEEncodeTiled", { pixels: ["4", 0], vae: ["5", 0], tile_size: 512, overlap: 128, temporal_size: 16, temporal_overlap: 8 }),','"7": node("VAEEncodeTiled", { pixels: ["4", 0], vae: ["5", 0], tile_size: tuning.tile, overlap: 128, temporal_size: tuning.temporal, temporal_overlap: 8 }),')
s=replace(s,'"chunking_mode.frames_per_chunk": 9 }),','"chunking_mode.frames_per_chunk": tuning.chunkFrames }),')
s=replace(s,'"12": node("VAEDecodeTiled", { samples: ["11", 0], vae: ["5", 0], tile_size: 512, overlap: 128, temporal_size: 16, temporal_overlap: 8 }),','"12": node("VAEDecodeTiled", { samples: ["11", 0], vae: ["5", 0], tile_size: tuning.tile, overlap: 128, temporal_size: tuning.temporal, temporal_overlap: 8 }),')
s=replace(s,'''  onStatus("SeedVR2 正在載入模型與提升畫質…");
  const result = await (await request("/prompt", {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ prompt: buildEnhancementWorkflow(source), client_id: crypto.randomUUID() }),''','''  onStatus("SeedVR2 正在載入模型與提升畫質…");
  const tuning = await enhancementTuning();
  const result = await (await request("/prompt", {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ prompt: buildEnhancementWorkflow(source, tuning), client_id: crypto.randomUUID() }),''')
s+='''
/** Bigger VAE tiles / temporal chunks on large GPUs: far fewer passes (defaults were sized for 12 GB laptops). */
export async function enhancementTuning(): Promise<EnhancementTuning> {
  try {
    const stats = await (await request("/system_stats")).json() as { devices?: { vram_total?: number }[] };
    const vram = (stats.devices?.[0]?.vram_total ?? 0) / 1024 ** 3;
    if (vram >= 38) return { tile: 1024, temporal: 32, chunkFrames: 21 };
    if (vram >= 20) return { tile: 768, temporal: 24, chunkFrames: 13 };
  } catch { /* fall back to the original conservative settings */ }
  return DEFAULT_TUNING;
}

/** System RAM in GiB, used to size segments (frames are held in RAM several times over). */
export async function systemRamGiB() {
  try {
    const stats = await (await request("/system_stats")).json() as { system?: { ram_total?: number } };
    return (stats.system?.ram_total ?? 0) / 1024 ** 3;
  } catch { return 0; }
}
'''
save(lib,s)
s=(REPO/ui).read_text(encoding='utf-8')
s=replace(s,'joinEnhancement, prepareEnhancement, prepareEnhancementFrom, runEnhancement } from "@/lib/enhance";','joinEnhancement, prepareEnhancement, prepareEnhancementFrom, runEnhancement, systemRamGiB } from "@/lib/enhance";')
s=replace(s,'      const chunk = tg >= 1440 ? 5 : tg >= 1080 ? 10 : 20;\n','      const ram = await systemRamGiB();\n      const chunk = tg >= 1440 ? (ram >= 100 ? 5 : 3) : tg >= 1080 ? (ram >= 60 ? 10 : 6) : 20;\n')
s=replace(s,'（720p 每段 20 秒、1080p 10 秒、1440p 5 秒）','（依機器記憶體自動決定每段長度）')
save(ui,s)
s=(REPO/enh).read_text(encoding='utf-8')
s=replace(s,'''"-map", "0:v:0", "-map", "0:a?", "-c:v", "libx264", "-crf", "18", "-preset", "fast",
                                 "-c:a", "aac", "-y", str(output)]''','''"-map", "0:v:0", "-map", "0:a?", "-c:v", "copy",
                                 "-c:a", "aac", "-y", str(output)]''')
save(enh,s)
# Enhancement speed 2: bicubic pre-upscale (PIL lanczos was single-core per frame) and 64px VAE tile overlap on big tiles.
s=(REPO/lib).read_text(encoding='utf-8')
s=replace(s,'"resize_type.crop": "disabled", scale_method: "lanczos" }),','"resize_type.crop": "disabled", scale_method: "bicubic" }),')
s=replace(s,'export type EnhancementTuning = { tile: number; temporal: number; chunkFrames: number };\nconst DEFAULT_TUNING: EnhancementTuning = { tile: 512, temporal: 16, chunkFrames: 9 };','export type EnhancementTuning = { tile: number; overlap: number; temporal: number; chunkFrames: number };\nconst DEFAULT_TUNING: EnhancementTuning = { tile: 512, overlap: 128, temporal: 16, chunkFrames: 9 };')
s=replace(s,'tile_size: tuning.tile, overlap: 128, temporal_size: tuning.temporal','tile_size: tuning.tile, overlap: tuning.overlap, temporal_size: tuning.temporal',2)
s=replace(s,'return { tile: 1024, temporal: 32, chunkFrames: 21 };','return { tile: 1024, overlap: 64, temporal: 32, chunkFrames: 21 };')
s=replace(s,'return { tile: 768, temporal: 24, chunkFrames: 13 };','return { tile: 768, overlap: 64, temporal: 24, chunkFrames: 13 };')
# Total processing time is saved with the result (shown on the card as 耗時 N 秒).
s=replace(s,'export async function blendEnhancement(source: EnhancementInput, raw: GeneratedVideo | null, strength: number) {','export async function blendEnhancement(source: EnhancementInput, raw: GeneratedVideo | null, strength: number, elapsed?: number) {')
s=replace(s,'    body: JSON.stringify({ sourceId: source.sourceId, raw, strength }),','    body: JSON.stringify({ sourceId: source.sourceId, raw, strength, elapsed }),')
s=replace(s,'export async function joinEnhancement(segments: string[], strength: number) {','export async function joinEnhancement(segments: string[], strength: number, elapsed?: number) {')
s=replace(s,'    body: JSON.stringify({ segments, strength }),','    body: JSON.stringify({ segments, strength, elapsed }),')
save(lib,s)
s=(REPO/ui).read_text(encoding='utf-8')
s=replace(s,'      const report = (text: string) => { if (sameSelection()) setStatus(text); onStatus(text); };\n','      const report = (text: string) => { if (sameSelection()) setStatus(text); onStatus(text); };\n      const began = Date.now();\n      const seconds = () => Math.max(1, Math.round((Date.now() - began) / 1000));\n')
s=replace(s,'const completed = await joinEnhancement(segments, chosenStrength);','const completed = await joinEnhancement(segments, chosenStrength, seconds());')
s=replace(s,'const completed = await blendEnhancement(current.input, current.raw, chosenStrength);','const completed = await blendEnhancement(current.input, current.raw, chosenStrength, seconds());')
save(ui,s)
s=(REPO/enh).read_text(encoding='utf-8')
s=replace(s,'def blend_video(source_id, raw, strength, segment=False):','def _seconds(elapsed):\n    return {"generationSeconds": max(1, round(float(elapsed)))} if isinstance(elapsed, (int, float)) and not isinstance(elapsed, bool) and math.isfinite(elapsed) and elapsed > 0 else {}\n\n\ndef blend_video(source_id, raw, strength, segment=False, elapsed=None):')
s=replace(s,'"duration": info["duration"], "sound": info["sound"], "extendable": False}','"duration": info["duration"], "sound": info["sound"], "extendable": False, **_seconds(elapsed)}')
s=replace(s,'bool(data.get("segment")))','bool(data.get("segment")), data.get("elapsed"))')
s=replace(s,'def join_segments(names, strength):','def join_segments(names, strength, elapsed=None):')
s=replace(s,'"extendable": False, "enhancementSegments": len(files)}','"extendable": False, "enhancementSegments": len(files), **_seconds(elapsed)}')
s=replace(s,'join_segments, data.get("segments"), data.get("strength"))','join_segments, data.get("segments"), data.get("strength"), data.get("elapsed"))')
save(enh,s)
# Segment length from a RAM budget (measured on Colab: 1440p x 3 s fit in 83 GB, 1080p x 10 s was OOM-killed at ~85 GB).
s=(REPO/ui).read_text(encoding='utf-8')
s=replace(s,'      const chunk = tg >= 1440 ? (ram >= 100 ? 5 : 3) : tg >= 1080 ? (ram >= 60 ? 10 : 6) : 20;\n','''      // ~0.25 GB of RAM per million output pixel-frames, plus ~15 GB for the loaded models; assume 16:9 at 30 fps.
      const budget = Math.max(20, (ram || 50) * 0.85 - 15) / 0.25 * 1e6;
      const chunk = Math.max(1, Math.min(20, Math.floor(budget / (tg * tg * 1.78 * 30) * 10) / 10));
''')
save(ui,s)
# Clear ComfyUI's node-output cache after every SeedVR2 job: each segment has new inputs, so old multi-GB frame tensors
# were kept in RAM and piled up across segments/jobs (1440p then 1080p was OOM-killed at ~85 GB).
s=(REPO/lib).read_text(encoding='utf-8')
s=replace(s,'''  onStatus("SeedVR2 正在載入模型與提升畫質…");
  const tuning = await enhancementTuning();''','''  onStatus("SeedVR2 正在載入模型與提升畫質…");
  // ComfyUI applies /free between queue items, checking at most every ~10 s; let a pending free land first.
  const wait = lastFree + 11000 - Date.now();
  if (wait > 0) await new Promise((resolve) => setTimeout(resolve, wait));
  const tuning = await enhancementTuning();''')
s=replace(s,'''  return waitForResult(result.prompt_id, () => onStatus("SeedVR2 正在提升畫質與 LAB 校色…"));''','''  try {
    return await waitForResult(result.prompt_id, () => onStatus("SeedVR2 正在提升畫質與 LAB 校色…"));
  } finally {
    await request("/free", { method: "POST", headers: { "Content-Type": "application/json" }, body: JSON.stringify({ free_memory: true }) }).catch(() => undefined);
    lastFree = Date.now();
  }''')
s=replace(s,'export type EnhancementTuning =','let lastFree = 0;\n\nexport type EnhancementTuning =')
save(lib,s)
# Optional FlashVSR engine for 畫質提升 (only when the FlashVSR node was installed in cell 6).
if globals().get('HAS_FLASHVSR'):
    s=(REPO/lib).read_text(encoding='utf-8')
    s+='''
export type EnhancementEngine = "seedvr2" | "flashvsr";

/** FlashVSR is trained for 4x: shrink to a quarter of the target (multiple of 32, so its 128-px output grid needs no crop),
 * upscale 4x, then resize to the exact target so blending/segment joins work unchanged. */
export function buildFlashWorkflow(source: EnhancementInput, vramGiB: number) {
  const node = (class_type: string, inputs: Record<string, unknown>) => ({ class_type, inputs });
  const quarter = (n: number) => Math.max(32, Math.round(n / 4 / 32) * 32);
  return {
    "1": node("LoadVideo", { file: source.inputFile }),
    "2": node("GetVideoComponents", { video: ["1", 0] }),
    "3": node("ResizeImageMaskNode", { input: ["2", 0], resize_type: "scale dimensions", "resize_type.width": quarter(source.outputWidth), "resize_type.height": quarter(source.outputHeight), "resize_type.crop": "disabled", scale_method: "area" }),
    "4": node("FlashVSRNode", { frames: ["3", 0], model: "FlashVSR-v1.1", mode: "tiny", scale: 4, tiled_vae: true, tiled_dit: vramGiB < 70, unload_dit: false, seed: 0 }),
    "5": node("ResizeImageMaskNode", { input: ["4", 0], resize_type: "scale dimensions", "resize_type.width": source.outputWidth, "resize_type.height": source.outputHeight, "resize_type.crop": "disabled", scale_method: "bicubic" }),
    "6": node("CreateVideo", { images: ["5", 0], fps: ["2", 2], bit_depth: 8 }),
    "7": node("SaveVideo", { video: ["6", 0], filename_prefix: `h3_enhance_cache/${source.sourceId}`, format: "mp4", "format.codec": "h264" }),
  };
}

async function vramGiB() {
  try {
    const stats = await (await request("/system_stats")).json() as { devices?: { vram_total?: number }[] };
    return (stats.devices?.[0]?.vram_total ?? 0) / 1024 ** 3;
  } catch { return 0; }
}
'''
    s=replace(s,'export async function runEnhancement(source: EnhancementInput, onStatus: (message: string) => void) {','export async function runEnhancement(source: EnhancementInput, onStatus: (message: string) => void, engine: EnhancementEngine = "seedvr2") {')
    s=replace(s,'  onStatus("SeedVR2 正在載入模型與提升畫質…");\n','  onStatus(engine === "flashvsr" ? "FlashVSR 正在載入模型與提升畫質（第一次會先下載模型）…" : "SeedVR2 正在載入模型與提升畫質…");\n')
    s=replace(s,'    body: JSON.stringify({ prompt: buildEnhancementWorkflow(source, tuning), client_id: crypto.randomUUID() }),','    body: JSON.stringify({ prompt: engine === "flashvsr" ? buildFlashWorkflow(source, await vramGiB()) : buildEnhancementWorkflow(source, tuning), client_id: crypto.randomUUID() }),')
    s=replace(s,'  if (!result.prompt_id) throw new Error(`SeedVR2 工作流無法執行：','  if (!result.prompt_id) throw new Error(`${engine === "flashvsr" ? "FlashVSR" : "SeedVR2"} 工作流無法執行：')
    s=replace(s,'    return await waitForResult(result.prompt_id, () => onStatus("SeedVR2 正在提升畫質與 LAB 校色…"));','    return await waitForResult(result.prompt_id, () => onStatus(engine === "flashvsr" ? "FlashVSR 正在提升畫質…" : "SeedVR2 正在提升畫質與 LAB 校色…"));')
    save(lib,s)
    s=(REPO/ui).read_text(encoding='utf-8')
    s=replace(s,'runEnhancement, systemRamGiB } from "@/lib/enhance";','runEnhancement, systemRamGiB, EnhancementEngine } from "@/lib/enhance";')
    s=replace(s,'function SeedVRStudio({ videos, connected, disabled, executionDisabled, onBusyChange, onComplete, enqueue }: {','function SeedVRStudio({ videos, connected, disabled, executionDisabled, onBusyChange, onComplete, enqueue, engine = "seedvr2" }: {')
    s=replace(s,'  enqueue: (label: string, run: (onStatus: (message: string) => void) => Promise<void>, onDiscard?: () => void) => number;\n}) {','  enqueue: (label: string, run: (onStatus: (message: string) => void) => Promise<void>, onDiscard?: () => void) => number;\n  engine?: EnhancementEngine;\n}) {')
    s=replace(s,'const chosenStrength = strength, src = source, seg = segment, st = start, du = duration, tg = target, token = selection.current;','const chosenStrength = strength, src = source, seg = segment, st = start, du = duration, tg = target, eng = engine, token = selection.current;')
    s=replace(s,'enqueue(`畫質提升 · ${name} · ${tg}p · ${chosenStrength}%`','enqueue(`畫質提升 · ${name} · ${tg}p · ${eng === "flashvsr" ? "FlashVSR" : "SeedVR2"} · ${chosenStrength}%`')
    s=replace(s,'const raw = chosenStrength > 0 ? await runEnhancement(input, (message) => report(`${tag}：${message}`)) : null;','const raw = chosenStrength > 0 ? await runEnhancement(input, (message) => report(`${tag}：${message}`), eng) : null;')
    s=replace(s,'current = { ...current, raw: await runEnhancement(current.input, report) };','current = { ...current, raw: await runEnhancement(current.input, report, eng) };')
    s=replace(s,'<option value="seedvr2">SeedVR2 · 提升至 720p</option>','<option value="seedvr2">SeedVR2 · 提升解析度（細節最多，較慢）</option><option value="flashvsr">FlashVSR · 提升解析度（快很多，畫面較平滑）</option>')
    s=replace(s,'<SeedVRStudio {...props}','<SeedVRStudio {...props} engine={method === "flashvsr" ? "flashvsr" : "seedvr2"}')
    save(ui,s)
# mk13: one 畫質處理方式 menu. Remount the SeedVR2 panel per method so a cached SeedVR2 result is never reused for FlashVSR.
s=(REPO/ui).read_text(encoding='utf-8')
s=replace(s,'<SeedVRStudio {...props}','<SeedVRStudio key={method} {...props}')
if not globals().get('HAS_FLASHVSR'):
    s=replace(s,'<option value="seedvr2">SeedVR2 · 提升至 720p</option>','<option value="seedvr2">SeedVR2 · 提升解析度</option>')
if not HAS_VOSR:
    s=replace(s,'<option value="vosr">VOSR · 1× 精修（維持原尺寸）</option>','')
if not HAS_SEED:
    s=replace(s,'useState(props.initialWork ? "vosr" : "seedvr2")','useState("vosr")')
    s,n=re.subn(r'<option value="(seedvr2|flashvsr)">[^<]*</option>','',s); assert n>=1
save(ui,s)
# VOSR: the worker runs with ComfyUI's own interpreter (Colab venv), not the Windows .venv path.
vosr='integrations/ComfyUI-H3-Studio/h3_studio_vosr.py'
s=pristine(vosr)
s=replace(s,'import subprocess\n','import subprocess\nimport sys\n')
s=replace(s,'root.parent / ".venv/Scripts/python.exe"','Path(sys.executable)',2)
save(vosr,s)
# Swap node: Colab runs without per-step cooldown, so accept 0 seconds.
sw='integrations/ComfyUI-H3-Studio/h3_studio_swap.py'
s=pristine(sw)
s=replace(s,'not 5 <= data["cooldownSeconds"] <= 600','not 0 <= data["cooldownSeconds"] <= 600')
s=replace(s,'每步降溫必須介於 5～600 秒。','每步降溫必須介於 0～600 秒。')
save(sw,s)
# mk19: head/character swap and face refine also load the Ref2VA main model; follow REF2VA_QUANT there too.
for rel in ('lib/swap.ts','lib/face-refine.ts'):
    p=WEB/rel; p.write_text(p.read_text(encoding='utf-8').replace('minimax_h3_ref2va_pruned_w4a8_mixed.safetensors',REF2VA_FILE),encoding='utf-8')
# Ref2VA: make sure no other Studio source still hardcodes the W4A8 filename.
if REF2VA_FILE!='minimax_h3_ref2va_pruned_w4a8_mixed.safetensors':
    leftovers=[str(p.relative_to(WEB)) for d in ('app','lib','components') if (WEB/d).is_dir() for p in (WEB/d).rglob('*') if p.suffix in ('.ts','.tsx','.js','.json') and 'minimax_h3_ref2va_pruned_w4a8_mixed.safetensors' in p.read_text(encoding='utf-8',errors='ignore')]
    assert not leftovers, '以下檔案仍寫死 W4A8 Ref2VA 檔名，需要補丁：'+', '.join(leftovers)
print('Ref2VA 工作流使用：',REF2VA_FILE)
# Build before launching; catches source or Linux dependency failures early.
run(['npm','run','build'],cwd=WEB)
print('Colab UI patched; original generation graphs and queue retained.')


Ref2VA 工作流使用： minimax_h3_ref2va_pruned_int8_convrot.safetensors
Colab UI patched; original generation graphs and queue retained.


In [9]:
# 8. 啟動原版服務，必須 HTTP 200 才算啟動成功。
import requests, requests.utils
# vinext production replies with zstd; urllib3 on Colab Py3.13 fails decoding chunked zstd. Only ask for gzip.
requests.utils.DEFAULT_ACCEPT_ENCODING='gzip, deflate'

DETACH="import subprocess,sys,json,os;p=subprocess.Popen(json.loads(sys.argv[1]),cwd=sys.argv[2],stdout=open(sys.argv[3],'w'),stderr=subprocess.STDOUT,stdin=subprocess.DEVNULL,start_new_session=True);print(p.pid,flush=True);os._exit(0)"
def launch_once(name,command,cwd,log,env=None):
    record=ROOT/(name+'.pid.json')
    if record.exists():
        saved=json.loads(record.read_text())
        if process_token(saved['pid'])==saved['token']:
            raise RuntimeError(name+' 已啟動；請先執行停止服務格，再重跑安裝／啟動')
        record.unlink()
    # Fully detached (own session, re-parented to init): interrupting/stopping a notebook cell
    # sends SIGTERM to the kernel's child processes, which previously took every service down.
    out=subprocess.run([sys.executable,'-c',DETACH,json.dumps([str(x) for x in command]),str(cwd),str(log)],capture_output=True,text=True,env=env,check=True,start_new_session=True)
    pid=int(out.stdout.strip()); time.sleep(0.3)
    token=process_token(pid)
    assert token is not None, name+' 立即退出，請查看 '+str(log)
    record.write_text(json.dumps({'pid':pid,'token':token}),encoding='utf-8')
    print('Started',name,pid)

def wait_url(url,seconds=300):
    until=time.monotonic()+seconds
    while time.monotonic()<until:
        try:
            r=requests.get(url,timeout=60)  # first SSR render can be slow
            if r.status_code==200: return r
        except requests.RequestException: pass
        time.sleep(2)
    raise RuntimeError('服務未就緒（只接受 HTTP 200）：'+url+'；查看下一格日誌')

service_env=os.environ.copy()
service_env['GEMINI_TTS_API_KEY']=GEMINI_KEY
launch_once('h3_comfy',[PYTHON,'main.py','--listen','127.0.0.1','--port','8188','--enable-cors-header','--disable-auto-launch','--extra-model-paths-config','extra_model_paths.yaml','--output-directory',OUTPUTS_ROOT],COMFY,ROOT/'h3_comfy.log',service_env)
stats=wait_url('http://127.0.0.1:8188/system_stats').json()
assert 'system' in stats and 'devices' in stats, '不是預期的 ComfyUI 服務'
# Production server (built in cell 7). No Vite dev server: avoids /@id/ 403, Host/Origin checks, slow on-demand compile.
launch_once('h3_studio',['npm','run','start','--','--hostname','127.0.0.1','--port','3000'],WEB,ROOT/'h3_studio.log')
wait_url('http://127.0.0.1:3000')
print('ComfyUI 與 Studio 已就緒')


Started h3_comfy 13518
Started h3_studio 13641
ComfyUI 與 Studio 已就緒


In [10]:
# 9. 檢查完整生成所需節點、模型、修臉狀態。缺少時不開公開網址。
base='http://127.0.0.1:8188'
r=requests.get(base+'/object_info',timeout=60); r.raise_for_status(); info=r.json()
required=['UNETLoader','CLIPLoader','VAELoader','MiniMaxH3TurboLoRA','MiniMaxH3TurboSampler','H3CooledTurboSampler','MiniMaxH3MemoryEfficientSageAttentionPatch','MiniMaxH3ReferenceToVideo','MiniMaxH3LatentLabLongMediaSetup','MiniMaxH3LatentLabLongMediaSampler','MiniMaxH3LatentLabLongMediaDecode','BlockSparseAttention','ModelAttentionBackend','SaveVideo','CreateVideo']
# Check every node class emitted by the two H3 graph builders, including cold-step/safe-long helpers.
source=(WEB/'lib/comfy.ts').read_text(encoding='utf-8')
for class_type in re.findall(r'node\("([A-Za-z0-9_]+)"',source):
    if class_type not in required: required.append(class_type)
missing=[name for name in required if name not in info]
assert not missing, '缺少生成節點：'+', '.join(missing)
unets=json.dumps(info['UNETLoader']['input']['required']['unet_name'])
assert REF2VA_FILE in unets, 'ComfyUI 看不到 Ref2VA 主模型：'+REF2VA_FILE
backend=info['ModelAttentionBackend']['input']['required']['attention']
assert 'comfy kitchen attention' in json.dumps(backend), 'Comfy Kitchen attention 未提供'
if INSTALL_FACE_REFINE:
    for query in ('','?mode=turbo-8&select=all_faces'):
        r=requests.get(base+'/h3-studio/face/status'+query,timeout=30); r.raise_for_status()
        ready=r.json(); assert ready.get('ready'), ready
if globals().get('HAS_FLASHVSR'): assert 'FlashVSRNode' in info, 'FlashVSR 節點未載入，請看 h3_comfy.log'
if globals().get('HAS_SINGULARITY'):
    assert SINGULARITY_MODEL in unets, 'ComfyUI 看不到 Singularity V2 主模型'
    assert SINGULARITY_LORA in json.dumps(info['LoraLoaderModelOnly']['input']['required']['lora_name']), 'ComfyUI 看不到 Singularity V2 加速 LoRA'
if globals().get('HAS_VOSR'):
    assert 'H3StudioVOSR' in info, 'H3StudioVOSR 節點未載入，請看 h3_comfy.log'
    r=requests.get(base+'/h3-studio/vosr/status',timeout=30); r.raise_for_status()
    assert r.json().get('ready'), ('VOSR 未就緒',r.json())
r=requests.get(base+'/h3-studio/outputs',timeout=30); r.raise_for_status()
print('節點與就緒狀態檢查通過；這尚未代替完整影片生成測試。')


節點與就緒狀態檢查通過；這尚未代替完整影片生成測試。


In [11]:
# 10. 成功的影片與圖片（Qwen 生圖等）自動複製到 Drive；原檔留在 /content。
# 不會複製失敗工作的未完成檔案；複製失敗會每 5 秒重試。
copy_script=r"""

from pathlib import Path
import json, os, shutil, time, urllib.request

SOURCE=Path('/content/h3_outputs').resolve()
DEST=Path('/content/drive/MyDrive/H3_Colab/outputs')
VIDEO_EXT={'.mp4','.webm','.mov','.png','.jpg','.jpeg','.webp'}  # mk21: images too
copied={}
RUN_TAG=time.strftime('%Y%m%d-%H%M%S')  # mk21: new runtimes restart Comfy's counter at 00001

def same_file(a,b):
    try:
        x,y=a.stat(),b.stat()
        return x.st_size==y.st_size and abs(x.st_mtime-y.st_mtime)<2
    except OSError:
        return False

def drive_target(video):
    # Drive path for this output; never overwrite a different earlier file with the same name.
    base=DEST/video.relative_to(SOURCE)
    if not base.exists() or same_file(video,base):
        return base
    for earlier in base.parent.glob(f'{base.stem}_*{base.suffix}'):
        if same_file(video,earlier):
            return earlier  # already renamed by an earlier run of this copier
    n=0
    while True:
        tag=RUN_TAG if n==0 else f'{RUN_TAG}-{n}'
        alt=base.with_name(f'{base.stem}_{tag}{base.suffix}')
        if not alt.exists() or same_file(video,alt):
            return alt
        n+=1

def fingerprint(video):
    meta=video.with_suffix(video.suffix+'.h3.json')
    return tuple((f.stat().st_size,f.stat().st_mtime_ns) if f.is_file() else None for f in (video,meta))

def history_files(value):
    if isinstance(value,dict):
        if value.get('type')=='output' and isinstance(value.get('filename'),str):
            yield SOURCE / value.get('subfolder','') / value['filename']
        for child in value.values():
            yield from history_files(child)
    elif isinstance(value,list):
        for child in value:
            yield from history_files(child)

while True:
    candidates=set()
    # Studio writes these sidecars only after final video processing succeeds.
    for meta in SOURCE.rglob('*.h3.json'):
        try:
            json.loads(meta.read_text(encoding='utf-8'))
            candidates.add(meta.with_name(meta.name[:-len('.h3.json')]))
        except (OSError,ValueError):
            pass
    # Also copy completed Comfy outputs if optional Studio metadata was not saved.
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/history?max_items=100',timeout=15) as response:
            history=json.load(response)
        for job in history.values():
            status=job.get('status',{})
            if status.get('completed') and status.get('status_str')=='success':
                candidates.update(history_files(job.get('outputs',{})))
    except Exception as exc:
        print('History check will retry:',type(exc).__name__,str(exc)[:160],flush=True)
    for candidate in sorted(candidates):
        temporary=[]
        try:
            video=candidate.resolve()
            relative=video.relative_to(SOURCE)
            if relative.parts and relative.parts[0]=='h3_enhance_cache':
                continue  # SeedVR2 intermediates; only the final enhanced video is backed up
            if video.suffix.lower() not in VIDEO_EXT or not video.is_file():
                continue
            before=fingerprint(video)
            if not before[0][0] or copied.get(video)==before:
                continue
            pairs=[]
            main_target=drive_target(video)
            for source in (video,video.with_suffix(video.suffix+'.h3.json')):
                if not source.is_file():
                    continue
                target=main_target if source==video else main_target.with_suffix(main_target.suffix+'.h3.json')
                target.parent.mkdir(parents=True,exist_ok=True)
                temp=target.with_name('.'+target.name+'.h3copy-part')
                temporary.append(temp)
                shutil.copy2(source,temp)
                if temp.stat().st_size!=source.stat().st_size:
                    raise OSError('Copy size mismatch')
                pairs.append((temp,target))
            if fingerprint(video)!=before:
                raise OSError('Source changed while copying; retrying')
            for temp,target in pairs:
                os.replace(temp,target)
            copied[video]=before
            note='' if main_target==DEST/relative else f' -> {main_target.name}（Drive 已有同名舊檔，改名保存）'
            print('Drive copy complete:',relative.as_posix()+note,flush=True)
        except Exception as exc:
            print('Drive copy will retry:',candidate,type(exc).__name__,str(exc)[:180],flush=True)
        finally:
            for temp in temporary:
                try:
                    temp.unlink(missing_ok=True)
                except OSError:
                    pass
    time.sleep(5)
"""
copy_path=ROOT/'h3_drive_copy.py'
copy_path.write_text(copy_script,encoding='utf-8')
launch_once('h3_drive_copy',[PYTHON,'-u',copy_path],ROOT,ROOT/'h3_drive_copy.log')
print('影片／圖片備份至:',DRIVE_OUTPUTS_ROOT)
print('請等 /content/h3_drive_copy.log 顯示 Drive copy complete，再刪除執行階段。')


Started h3_drive_copy 13687
影片／圖片備份至: /content/drive/MyDrive/H3_Colab/outputs
請等 /content/h3_drive_copy.log 顯示 Drive copy complete，再刪除執行階段。


In [12]:
# 11. 密碼保護的公開網址。先取得 tunnel 域名，再建立嚴格的 Origin 檢查。
launch_once('h3_cloudflared',['/usr/local/bin/cloudflared','tunnel','--no-autoupdate','--protocol','http2','--url','http://127.0.0.1:8081'],ROOT,ROOT/'h3_cloudflared.log')
public_url=None
for _ in range(90):
    text=(ROOT/'h3_cloudflared.log').read_text(errors='replace')
    m=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',text)
    if m: public_url=m.group(0); break
    time.sleep(2)
assert public_url, '未取得 tunnel 網址，請查看 h3_cloudflared.log'
# stdin prevents putting the password in process arguments or notebook outputs.
subprocess.run(['htpasswd','-iBc','/content/.h3_basic_auth',BASIC_USER],input=BASIC_PASSWORD+'\n',text=True,check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
# Only loopback listeners. Public Host is rewritten for Vite; browser Origin is validated first.
conf=r"""
map $http_origin $h3_origin_ok {
 default 0;
 "" 1;
 "__PUBLIC_URL__" 1;
}
server {
 listen 127.0.0.1:8081;
 server_name _;
 client_max_body_size 2048m;
 auth_basic "H3 Colab Studio";
 auth_basic_user_file /content/.h3_basic_auth;
 proxy_read_timeout 3600s;
 proxy_send_timeout 3600s;
 # These features are explicitly disabled for the Colab edition.
 location = /comfy/h3-studio/output { return 405; }
 location = /comfy/h3-studio/concat { return 405; }
 location = /comfy/h3-studio/continuation-frame { return 405; }
 location = /comfy/h3-studio/optimize-prompt { return 501; }
 location = /comfy/h3-studio/optimize-image-prompt { return 501; }
 location /comfy/h3-studio/drama3/ { return 501; }
 location /comfy/ {
  if ($h3_origin_ok = 0) { return 403; }
  proxy_pass http://127.0.0.1:8188/;
  proxy_http_version 1.1;
  proxy_set_header Host 127.0.0.1:8188;
  proxy_set_header Origin http://127.0.0.1:3000;
  proxy_set_header Upgrade $http_upgrade;
  proxy_set_header Connection "upgrade";
 }
 location / {
  if ($h3_origin_ok = 0) { return 403; }
  proxy_pass http://127.0.0.1:3000;
  proxy_http_version 1.1;
  proxy_set_header Host 127.0.0.1:3000;
  # Dev server rejects module/RSC requests whose Origin != Host (403). Origin already validated above.
  proxy_set_header Origin http://127.0.0.1:3000;
  proxy_set_header X-Forwarded-Proto https;
  proxy_set_header Upgrade $http_upgrade;
  proxy_set_header Connection "upgrade";
 }
}
""".replace('__PUBLIC_URL__',public_url)
Path('/etc/nginx/conf.d/h3.conf').write_text(conf,encoding='utf-8')
# Colab nginx map hostname hash requires larger buckets (verified via nginx -t).
nginx_conf=Path('/etc/nginx/nginx.conf')
nginx_text=nginx_conf.read_text(encoding='utf-8')
if 'map_hash_bucket_size' not in nginx_text:
    assert nginx_text.count('http {') == 1, 'Unexpected nginx.conf format'
    nginx_text=nginx_text.replace('http {','http {\n    map_hash_bucket_size 128;',1)
    nginx_conf.write_text(nginx_text,encoding='utf-8')
default=Path('/etc/nginx/sites-enabled/default')
if default.is_symlink(): default.unlink()
run(['nginx','-t'])
if subprocess.run(['nginx','-s','reload'],capture_output=True).returncode: run(['nginx'])
# reload is async: old workers keep the previous tunnel URL for a moment -> spurious 403.
time.sleep(3)
# Prove the auth/host/origin chain works before displaying the link.
local='http://127.0.0.1:8081'
auth=(BASIC_USER,BASIC_PASSWORD)
assert requests.get(local,timeout=30).status_code==401, '公開代理沒有要求密碼'
for route in ('/','/comfy/system_stats','/comfy/h3-studio/tts/status'):
    r=requests.get(local+route,auth=auth,headers={'Host':public_url.split('://')[1],'Origin':public_url},timeout=60)
    assert r.status_code==200, (route,r.status_code,r.text[:200])
assert requests.get(local+'/comfy/h3-studio/tts/status',auth=auth,headers={'Origin':'https://invalid.example'},timeout=30).status_code==403
for _ in range(20):
    try:
        r=requests.get(public_url,auth=auth,timeout=20)
        if r.status_code==200: break
    except requests.RequestException: pass
    time.sleep(2)
else:
    raise RuntimeError('本機代理已通過，但公開 tunnel 未就緒；請查看日誌，不要提交生成工作')
from IPython.display import display, Markdown
display(Markdown('### [開啟 H3 Colab Studio]('+public_url+')\n帳號：`'+BASIC_USER+'`；密碼為第 1 格設定值。'))
print('首次請先生成短片，確認畫面、聲音與 Drive 備份，再增加片長。')


Started h3_cloudflared 13693


### [開啟 H3 Colab Studio](https://detect-emphasis-hundreds-cod.trycloudflare.com)
帳號：`h3`；密碼為第 1 格設定值。

首次請先生成短片，確認畫面、聲音與 Drive 備份，再增加片長。


In [ ]:
# 11b. 生成期間保持 notebook 活躍，並每分鐘回報所有服務狀態（不用另外跑檢查指令）。
# 服務已完全脫離 notebook，按停止這格不會影響 ComfyUI／Studio／tunnel。
import time, json, re, requests
def _alive(name):
    rec=ROOT/(name+'.pid.json')
    try: saved=json.loads(rec.read_text()); return process_token(saved['pid'])==saved['token']
    except Exception: return False
def _ok(url):
    try: return requests.get(url,timeout=10).status_code==200
    except Exception: return False
m=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',(ROOT/'h3_cloudflared.log').read_text(errors='replace')) if (ROOT/'h3_cloudflared.log').exists() else None
print('目前 Studio 網址：',m.group(0) if m else '（找不到，請重跑第 11 格或快速恢復）',flush=True)
try:
    while True:
        if not _alive('h3_drive_copy'):
            (ROOT/'h3_drive_copy.pid.json').unlink(missing_ok=True)
            launch_once('h3_drive_copy',[PYTHON,'-u',ROOT/'h3_drive_copy.py'],ROOT,ROOT/'h3_drive_copy.log')
            print('⚠ Drive 備份程式已停止，已自動重啟',flush=True)
        try: q=requests.get('http://127.0.0.1:8188/queue',timeout=10).json(); busy=len(q.get('queue_running',[]))+len(q.get('queue_pending',[]))
        except Exception: busy='?'
        log=ROOT/'h3_drive_copy.log'
        last=log.read_text(errors='replace').strip().splitlines()[-1:] if log.exists() else []
        flag=lambda b:'OK' if b else '✗'
        print(time.strftime('%H:%M'),f"ComfyUI {flag(_ok('http://127.0.0.1:8188/system_stats'))}",f"Studio {flag(_ok('http://127.0.0.1:3000'))}",
              f"tunnel {flag(_alive('h3_cloudflared'))}",'佇列:',busy,'|',last[0][-80:] if last else '尚無備份紀錄',flush=True)
        time.sleep(60)
except KeyboardInterrupt:
    print('已停止保持活躍（服務仍在背景執行）')


目前 Studio 網址： https://detect-emphasis-hundreds-cod.trycloudflare.com
01:31 ComfyUI OK Studio OK tunnel OK 佇列: 0 | 尚無備份紀錄
01:32 ComfyUI OK Studio OK tunnel OK 佇列: 0 | 尚無備份紀錄
01:33 ComfyUI OK Studio OK tunnel OK 佇列: 0 | 尚無備份紀錄
01:34 ComfyUI OK Studio OK tunnel OK 佇列: 0 | 尚無備份紀錄
01:35 ComfyUI OK Studio OK tunnel OK 佇列: 1 | 尚無備份紀錄
01:36 ComfyUI OK Studio OK tunnel OK 佇列: 1 | 尚無備份紀錄
01:37 ComfyUI OK Studio OK tunnel OK 佇列: 1 | 尚無備份紀錄
01:38 ComfyUI OK Studio OK tunnel OK 佇列: 1 | 尚無備份紀錄
01:39 ComfyUI OK Studio OK tunnel OK 佇列: 1 | 尚無備份紀錄
01:40 ComfyUI OK Studio OK tunnel OK 佇列: 1 | 尚無備份紀錄
01:41 ComfyUI OK Studio OK tunnel OK 佇列: 1 | 尚無備份紀錄
01:42 ComfyUI OK Studio OK tunnel OK 佇列: 0 | Drive copy complete: H3_Image/H3_Studio_QWEN_00001_.png
01:43 ComfyUI OK Studio OK tunnel OK 佇列: 0 | Drive copy complete: H3_Image/H3_Studio_QWEN_00001_.png
01:44 ComfyUI OK Studio OK tunnel OK 佇列: 0 | Drive copy complete: H3_Image/H3_Studio_QWEN_00001_.png
01:45 ComfyUI OK Studio OK tunnel OK 佇列: 0 | Drive copy 

In [ ]:
# R. 快速恢復（kernel 重啟／當機後用）：不重裝、不重 build，直接重啟所有服務並給新網址。
# 前提：這個執行階段之前已完整跑過第 1～11 格。
from google.colab import drive
from pathlib import Path
import os, sys, json, signal, time, re, subprocess, getpass
drive.mount('/content/drive')
ROOT=Path('/content'); DRIVE_ROOT=ROOT/'drive/MyDrive/H3_Colab'; OUTPUTS_ROOT=ROOT/'h3_outputs'; DRIVE_OUTPUTS_ROOT=DRIVE_ROOT/'outputs'
REPO=ROOT/'LocalStudio'; COMFY=ROOT/'ComfyUI'; WEB=REPO/'h3-local-studio'; VENV=ROOT/'h3_venv'; PYTHON=VENV/'bin/python'
BASIC_USER='h3'
while True:
    BASIC_PASSWORD=getpass.getpass('Studio 密碼（至少 8 字元；登入帳號是 h3）：')
    if len(BASIC_PASSWORD)>=8: break
    print(f'⚠ 密碼只有 {len(BASIC_PASSWORD)} 個字元，至少要 8 個，請重新輸入。',flush=True)
GEMINI_KEY=getpass.getpass('Gemini TTS API key（選填，Enter 略過）：').strip()
import requests, requests.utils
requests.utils.DEFAULT_ACCEPT_ENCODING='gzip, deflate'
def run(args,**kw): subprocess.run([str(x) for x in args],check=True,**kw)
def process_token(pid):
    try:
        fields=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
        return None if fields[0]=='Z' else fields[19]
    except (FileNotFoundError,ProcessLookupError): return None
def stop_managed(name):
    record=ROOT/(name+'.pid.json')
    if not record.exists(): return
    saved=json.loads(record.read_text()); pid=saved['pid']
    if process_token(pid)==saved['token']:
        os.killpg(pid,signal.SIGTERM)
        for _ in range(50):
            if process_token(pid)!=saved['token']: break
            time.sleep(0.1)
        if process_token(pid)==saved['token']: os.killpg(pid,signal.SIGKILL)
    record.unlink(missing_ok=True)
DETACH="import subprocess,sys,json,os;p=subprocess.Popen(json.loads(sys.argv[1]),cwd=sys.argv[2],stdout=open(sys.argv[3],'w'),stderr=subprocess.STDOUT,stdin=subprocess.DEVNULL,start_new_session=True);print(p.pid,flush=True);os._exit(0)"
def launch_once(name,command,cwd,log,env=None):
    record=ROOT/(name+'.pid.json')
    if record.exists():
        saved=json.loads(record.read_text())
        if process_token(saved['pid'])==saved['token']: raise RuntimeError(name+' 已啟動')
        record.unlink()
    out=subprocess.run([sys.executable,'-c',DETACH,json.dumps([str(x) for x in command]),str(cwd),str(log)],capture_output=True,text=True,env=env,check=True,start_new_session=True)
    pid=int(out.stdout.strip()); time.sleep(0.3)
    token=process_token(pid); assert token, name+' 立即退出，請看 '+str(log)
    record.write_text(json.dumps({'pid':pid,'token':token}),encoding='utf-8'); print('Started',name,pid)
def wait_url(url,seconds=300):
    until=time.monotonic()+seconds
    while time.monotonic()<until:
        try:
            r=requests.get(url,timeout=60)
            if r.status_code==200: return r
        except requests.RequestException: pass
        time.sleep(2)
    raise RuntimeError('服務未就緒：'+url)
# Keep the previous logs (快速恢復 overwrites them) for post-mortem.
import shutil
for n in ('h3_studio','h3_comfy','h3_drive_copy','h3_cloudflared'):
    if (ROOT/(n+'.log')).exists(): shutil.copy(ROOT/(n+'.log'),ROOT/(n+'.log.prev'))
for name in ('h3_cloudflared','h3_drive_copy','h3_studio','h3_comfy'): stop_managed(name)
assert (WEB/'dist').is_dir() and (ROOT/'h3_drive_copy.py').is_file(), '這個執行階段沒有完整跑過第 1～11 格，請從頭執行'
service_env=os.environ.copy(); service_env['GEMINI_TTS_API_KEY']=GEMINI_KEY
launch_once('h3_comfy',[PYTHON,'main.py','--listen','127.0.0.1','--port','8188','--enable-cors-header','--disable-auto-launch','--extra-model-paths-config','extra_model_paths.yaml','--output-directory',OUTPUTS_ROOT],COMFY,ROOT/'h3_comfy.log',service_env)
wait_url('http://127.0.0.1:8188/system_stats')
launch_once('h3_studio',['npm','run','start','--','--hostname','127.0.0.1','--port','3000'],WEB,ROOT/'h3_studio.log')
wait_url('http://127.0.0.1:3000')
launch_once('h3_drive_copy',[PYTHON,'-u',ROOT/'h3_drive_copy.py'],ROOT,ROOT/'h3_drive_copy.log')
launch_once('h3_cloudflared',['/usr/local/bin/cloudflared','tunnel','--no-autoupdate','--protocol','http2','--url','http://127.0.0.1:8081'],ROOT,ROOT/'h3_cloudflared.log')
public_url=None
for _ in range(90):
    m=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',(ROOT/'h3_cloudflared.log').read_text(errors='replace'))
    if m: public_url=m.group(0); break
    time.sleep(2)
assert public_url, '未取得 tunnel 網址'
subprocess.run(['htpasswd','-iBc','/content/.h3_basic_auth',BASIC_USER],input=BASIC_PASSWORD+'\n',text=True,check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
conf=Path('/etc/nginx/conf.d/h3.conf'); text=conf.read_text(encoding='utf-8')
text=re.sub(r'"https://[a-zA-Z0-9-]+\.trycloudflare\.com" 1;','"'+public_url+'" 1;',text); conf.write_text(text,encoding='utf-8')
run(['nginx','-t'])
if subprocess.run(['nginx','-s','reload'],capture_output=True).returncode: run(['nginx'])
time.sleep(3)
local='http://127.0.0.1:8081'; auth=(BASIC_USER,BASIC_PASSWORD); host=public_url.split('://')[1]
for _ in range(15):
    if requests.get(local+'/',auth=auth,headers={'Host':host,'Origin':public_url},timeout=60).status_code==200: break
    time.sleep(2)
else: raise RuntimeError('本機代理檢查失敗')
for _ in range(20):
    try:
        if requests.get(public_url,auth=auth,timeout=20).status_code==200: break
    except requests.RequestException: pass
    time.sleep(2)
from IPython.display import display, Markdown
display(Markdown('### [開啟 H3 Colab Studio]('+public_url+')\n帳號：`'+BASIC_USER+'`'))
print('已恢復。接著執行 11b 保持活躍。')


In [ ]:
# 12. 查看日誌與尚未備份的已完成影片／圖片（可重複執行，不送 GPU 工作）。
for name in ('h3_comfy','h3_studio','h3_cloudflared','h3_drive_copy'):
    log=ROOT/(name+'.log')
    print('\n===',name,'===')
    print(log.read_text(errors='replace')[-4000:] if log.exists() else 'not started')
pending=[]
for meta in OUTPUTS_ROOT.rglob('*.h3.json'):
    video=meta.with_name(meta.name[:-len('.h3.json')])
    if video.suffix.lower() not in {'.mp4','.webm','.mov','.png','.jpg','.jpeg','.webp'} or not video.is_file(): continue
    base=DRIVE_OUTPUTS_ROOT/video.relative_to(OUTPUTS_ROOT)
    def _done(target):
        target_meta=target.with_suffix(target.suffix+'.h3.json')
        return target.is_file() and target.stat().st_size==video.stat().st_size and target_meta.is_file() and target_meta.read_bytes()==meta.read_bytes()
    # Renamed copies (name_YYYYMMDD-HHMMSS.ext) count too.
    if not (_done(base) or any(_done(x) for x in base.parent.glob(base.stem+'_*'+base.suffix))):
        pending.append(str(video.relative_to(OUTPUTS_ROOT)))
print('尚未確認備份完成:',pending if pending else '沒有（仍請查看 copy 日誌）')


## 停止服務（需要停止時才執行）
停止前先執行上一格，等所有影片顯示 `Drive copy complete`，確認 Drive 中能開啟。停止服務不等於刪除 Colab 執行階段，也不代表停止 CU 計費；不使用時請在 Colab 選「中斷連線並刪除執行階段」。


In [ ]:
for name in ('h3_cloudflared','h3_drive_copy','h3_studio','h3_comfy'):
    stop_managed(name)
print('已停止 notebook 管理的服務；Drive 備份不會刪除。')
